# Runner econométrico — Credit Spreads S&P 500

## Objetivo

Este notebook implementa la estrategia econométrica final de la tesis, cuyo objetivo es analizar cómo los credit spreads de bonos corporativos del S&P 500 reflejan:

1. condiciones macro-financieras comunes,
2. shocks agregados de mercado y crédito,
3. heterogeneidad en la exposición firm-level a dichos shocks.

---

## Estrategia empírica

La estimación sigue una arquitectura híbrida:

- **Backbone + módulos (Arquitectura B):**
  se modelan explícitamente los factores agregados observables.

- **Secuencia M0–M6 (Arquitectura A):**
  los modelos se presentan progresivamente para aislar canales económicos.

- **Robustez en variable dependiente (Arquitectura C):**
  se evalúa la estabilidad de los resultados usando distintas medidas de credit spread.

---

## Variable dependiente

- Principal: `oas_mean`
- Robustez:
  - `spread_mean_bps`
  - `cds_5y_mean` o `cds_5y_eom`

---

## Estructura del notebook

1. Setup y configuración  
2. Carga del panel final  
3. Definición conceptual de variables  
4. Preparación de variables  
5. Construcción de interacciones  
6. Definición de muestras  
7. Funciones de estimación  
8. Modelos M0–M6  
9. Estimación principal (OAS)  
10. Interpretación económica  
11. Export de resultados  
12. Robustez (dependiente)  
13. Robusteces adicionales  
14. QA final  
15. Cierre  

---

## Principios clave

- Efectos fijos por firma en todos los modelos  
- Efectos fijos de tiempo solo en el benchmark (M0)  
- Errores clusterizados por firma  
- `ivol_252` se incluye siempre y no se interactúa  
- Market power se utiliza solo como modulador (interacciones)  
- Separación explícita entre:
  - shocks agregados
  - exposición diferencial

## 1) Setup y configuración

En esta sección se definen:

- librerías necesarias,
- paths del proyecto,
- configuración de warnings y display,
- directorios de output.

El objetivo es asegurar que el notebook sea completamente reproducible y consistente con la estructura del repositorio.

In [15]:
# ==========================================================
# 1. SETUP Y CONFIGURACIÓN
# ==========================================================

# ------------------------------
# Librerías
# ------------------------------
import pandas as pd
import numpy as np

from pathlib import Path
import warnings

# Modelos econométricos
from linearmodels.panel import PanelOLS

# ------------------------------
# Configuración general
# ------------------------------
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ------------------------------
# Paths del proyecto
# ------------------------------
BASE_DIR = Path.cwd().resolve().parents[0]

DATA_DIR = BASE_DIR / "data"
CLEAN_DIR = DATA_DIR / "clean"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
RESULTS_DIR = OUTPUT_DIR / "results"

# Crear carpetas si no existen
TABLES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# Parámetros globales
# ------------------------------
CLUSTER_ENTITY = True   # cluster por firma
USE_TIME_FE_M0 = True   # FE tiempo solo en M0

print("Setup completo.")


Setup completo.


## 2) Carga del panel final

En esta sección se carga el panel final construido en el Notebook 2.

El panel contiene información a nivel firma-tiempo e incluye:

- variables dependientes (OAS, spreads, CDS),
- shocks agregados (mercado y crédito),
- variables macroeconómicas,
- controles firm-level,
- proxies de market power.

---

## Validaciones realizadas

Se realizan chequeos básicos para asegurar:

- correcta carga del archivo,
- dimensiones del panel,
- presencia de columnas clave,
- consistencia de tipos de datos,
- unicidad de identificador firma-tiempo.

---

## Output esperado

Un DataFrame limpio, consistente y listo para:

- definición de variables,
- construcción de muestras,
- estimación econométrica.

### Estandarización de nombres de variables

Se renombran columnas para evitar problemas en la estimación:

- se eliminan espacios,
- se reemplazan caracteres especiales,
- se homogeniza formato snake_case.

Esto es necesario para compatibilidad con linearmodels y fórmulas econométricas.

In [16]:
# ==========================================================
# 2. CARGA DEL PANEL FINAL
# ==========================================================

# ------------------------------
# Path del panel
# ------------------------------
PANEL_PATH = CLEAN_DIR / "panel_master.parquet"

# ------------------------------
# Carga
# ------------------------------
print("Cargando panel desde:", PANEL_PATH)
df = pd.read_parquet(PANEL_PATH)
print("Panel cargado correctamente.")

# ------------------------------
# Dimensiones iniciales
# ------------------------------
print("\nDimensión del panel:")
print(df.shape)

# ------------------------------
# Columnas disponibles (raw)
# ------------------------------
print("\nColumnas del panel (raw):")
print(sorted(df.columns.tolist()))

# ------------------------------
# Estandarización de nombres de columnas
# ------------------------------
def clean_column_names(columns):
    return (
        columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

df.columns = clean_column_names(df.columns)

print("\nColumnas estandarizadas:")
print(sorted(df.columns.tolist()))

# ------------------------------
# Tipos de datos
# ------------------------------
print("\nTipos de datos:")
print(df.dtypes.head(20))

# ------------------------------
# Conversión de fecha
# ------------------------------
if "date" not in df.columns:
    raise ValueError("No existe la columna 'date' en el panel.")

if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ------------------------------
# Chequeo de identificadores clave
# ------------------------------
required_ids = ["issuer", "date"]
missing_ids = [c for c in required_ids if c not in df.columns]

if missing_ids:
    raise ValueError(f"Faltan columnas clave del panel: {missing_ids}")

# ------------------------------
# Unicidad firma-tiempo
# ------------------------------
duplicates = df.duplicated(subset=["issuer", "date"]).sum()
print(f"\nDuplicados (issuer-date): {duplicates}")

# ------------------------------
# Orden del panel
# ------------------------------
df = df.sort_values(["issuer", "date"]).reset_index(drop=True)

# ------------------------------
# Chequeo de valores faltantes clave
# ------------------------------
key_vars = [
    "oas_mean",
    "spread_mean_bps",
    "cds_5y_mean",
    "mkt_ret",
    "credit_level",
    "credit_slope"
]

missing_key_vars = [c for c in key_vars if c not in df.columns]
if missing_key_vars:
    raise ValueError(f"Faltan variables clave del panel: {missing_key_vars}")

missing_summary = df[key_vars].isna().mean().sort_values(ascending=False)

print("\n% de missing en variables clave:")
print(missing_summary)

# ------------------------------
# Chequeo de market power
# ------------------------------
market_power_vars = [
    "market_share_raw",
    "market_share_w",
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
]

available_mp = [c for c in market_power_vars if c in df.columns]
missing_mp = [c for c in market_power_vars if c not in df.columns]

print("\nProxies de market power disponibles:")
print(available_mp)

if missing_mp:
    print("\nProxies de market power faltantes:")
    print(missing_mp)

if available_mp:
    print("\n% de missing en proxies de market power:")
    print(df[available_mp].isna().mean().sort_values(ascending=False))

# ------------------------------
# Chequeo rápido de panel
# ------------------------------
n_firms = df["issuer"].nunique()
n_periods = df["date"].nunique()

print("\nEstructura del panel:")
print(f"- Firmas: {n_firms}")
print(f"- Periodos: {n_periods}")
print(f"- Observaciones totales: {len(df)}")

print("\nCarga y validación inicial completadas.")

Cargando panel desde: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data\clean\panel_master.parquet
Panel cargado correctamente.

Dimensión del panel:
(28701, 68)

Columnas del panel (raw):
['US1M', 'US1Y', 'US30Y', 'US3Y', 'US5Y', 'US6M', 'US7Y', 'beta_252', 'cash_to_assets', 'cds_5y_eom', 'cds_5y_mean', 'covid_dummy', 'crc_level_beta', 'crc_nobs', 'crc_r2', 'crc_slope_beta', 'credit_curve_change', 'credit_curve_change_l1', 'credit_curve_level', 'credit_curve_level_l1', 'credit_curve_shock', 'credit_curve_shock_l1', 'credit_level', 'credit_long_return', 'credit_market_level_log', 'credit_market_level_log_l1', 'credit_market_return', 'credit_short_return', 'credit_slope', 'credit_tightening_shock', 'credit_tightening_shock_l1', 'current_ratio', 'date', 'interest_coverage', 'issuer', 'ivol_252', 'ivol_sector', 'leverage', 'log_assets', 'ltdebt_share', 'market_share_industry', 'market_share_industry_group', 'market_share_raw', 'market_share_sector', 'market_share_subindustry', '

In [17]:
print("\n" + "="*60)
print("DEBUG MARKET POWER")
print("="*60)

# 1) Ver todas las columnas
print("\nColumnas en df:")
print(df.columns.tolist())

# 2) Buscar columnas candidatas relacionadas a market power
print("\nColumnas que contienen 'market', 'share', 'power':")
candidates = [c for c in df.columns if any(x in c.lower() for x in ["market", "share", "power"])]
print(candidates)

# 3) Chequeo directo de la variable esperada
print("\n¿Existe 'market_share_industry'?")
print("market_share_industry" in df.columns)

# 4) Si existe, ver contenido
if "market_share_industry" in df.columns:
    print("\nHead de market_share_industry:")
    print(df["market_share_industry"].head())

    print("\nDescribe:")
    print(df["market_share_industry"].describe())

    print("\nNaN share:")
    print(df["market_share_industry"].isna().mean())

# 5) Chequear otras posibles variables de market power típicas
possible_vars = [
    "market_share",
    "market_share_sector",
    "market_share_industry_group",
    "hhi",
    "hhi_industry",
    "relative_size",
    "markup",
]

print("\nChequeo de variables alternativas posibles:")
for v in possible_vars:
    print(f"{v}: {v in df.columns}")


DEBUG MARKET POWER

Columnas en df:
['issuer', 'date', 'spread_mean_bps', 'ytm_mean', 'residual_maturity_mean', 'n_bonds', 'ticker', 'ticker_base', 'sector', 'beta_252', 'ivol_252', 'log_assets', 'leverage', 'cash_to_assets', 'current_ratio', 'interest_coverage', 'ltdebt_share', 'rollover_12m', 'market_share_raw', 'market_share_w', 'market_share_sector', 'market_share_industry_group', 'market_share_industry', 'market_share_subindustry', 'crc_level_beta', 'crc_slope_beta', 'crc_r2', 'crc_nobs', 'mkt_ret', 'mkt_vol_60d', 'credit_level', 'credit_slope', 'credit_tightening_shock', 'credit_curve_shock', 'credit_market_return', 'credit_short_return', 'credit_long_return', 'credit_market_level_log', 'credit_curve_level', 'credit_curve_change', 'credit_tightening_shock_l1', 'credit_curve_shock_l1', 'credit_market_level_log_l1', 'credit_curve_level_l1', 'credit_curve_change_l1', 'policy_rate', 'term_spread_10y_2y', 'move_eom', 'ust_3m', 'ust_2y', 'ust_10y', 'term_spread_10y_3m', 'us1m', 'us1y'

## 3) Definición conceptual de variables

Esta sección define la estructura económica del modelo empírico.

La especificación distingue explícitamente entre:

1. Variable dependiente
2. Controles firm-level
3. Riesgo idiosincrático (zero-beta)
4. Entorno macro-financiero (backbone)
5. Shocks agregados (mercado y crédito)
6. Heterogeneidad en la exposición (interacciones)

---

## Principio de identificación

La estrategia empírica separa tres componentes fundamentales:

- Factores agregados comunes (macro)
- Shocks agregados observables (mercado y crédito)
- Exposición diferencial entre firmas

Esto permite evitar interpretaciones erróneas en las que variables firm-level capturan shocks agregados.

---

## Variable dependiente

- Principal: `oas_mean`
- Robustez:
  - `spread_mean_bps`
  - `cds_5y_mean` / `cds_5y_eom`

---

## Bloques del modelo

### Controles firm-level
Capturan características estructurales de las firmas.

### Riesgo idiosincrático
`ivol_252` se incluye como componente zero-beta.

### Backbone macro-financiero
Variables agregadas que reemplazan efectos fijos de tiempo.

### Shocks agregados
Capturan innovaciones en el entorno de mercado y crédito.

### Heterogeneidad
Interacciones que permiten exposición diferencial a shocks.

In [18]:
# ==========================================================
# 3.1 VARIABLE DEPENDIENTE
# ==========================================================

DEP_VAR_MAIN = "oas_mean"

DEP_VARS_ALT = [
    "spread_mean_bps",
    "cds_5y_mean",   # alternativa posible más adelante: cds_5y_eom
]

# ==========================================================
# 3.2 CONTROLES FIRM-LEVEL
# ==========================================================

FIRM_CONTROLS = [
    "leverage",
    "log_assets",
    "cash_to_assets",
    "current_ratio",
    "interest_coverage",
    "residual_maturity_mean",
    "rollover_12m"
]

missing_controls = [c for c in FIRM_CONTROLS if c not in df.columns]
if missing_controls:
    raise ValueError(f"Faltan controles firm-level ya construidos en el panel: {missing_controls}")

print("✔ Controles firm-level correctamente detectados")

# ==========================================================
# 3.3 RIESGO IDIOSINCRÁTICO
# ==========================================================

IVOL_VAR = "ivol_252"

if IVOL_VAR not in df.columns:
    raise ValueError(f"Falta variable de riesgo idiosincrático: {IVOL_VAR}")

print("✔ Riesgo idiosincrático correctamente detectado")

# ==========================================================
# 3.4 BACKBONE MACRO-FINANCIERO
# ==========================================================

MACRO_VARS = [
    "policy_rate",
    "term_spread_10y_2y",
    "move_eom"
]

MACRO_VARS_COVID = MACRO_VARS + ["covid_dummy"]

missing_macro = [c for c in MACRO_VARS if c not in df.columns]
if missing_macro:
    raise ValueError(f"Faltan variables macro-financieras: {missing_macro}")

print("✔ Backbone macro-financiero correctamente detectado")

# ==========================================================
# 3.5 SHOCKS AGREGADOS
# ==========================================================

MARKET_SHOCK = "mkt_ret"

# ------------------------------
# Crédito: especificación legacy
# ------------------------------
CREDIT_VARS_LEGACY = [
    "credit_level",
    "credit_slope"
]

# ------------------------------
# Crédito: nombres más claros
# ------------------------------
CREDIT_VARS_SHOCK = [
    "credit_tightening_shock",
    "credit_curve_shock"
]

# ------------------------------
# Crédito: shocks laggeados
# ------------------------------
CREDIT_VARS_SHOCK_L1 = [
    "credit_tightening_shock_l1",
    "credit_curve_shock_l1"
]

# ------------------------------
# Crédito: niveles / alternativas nuevas
# ------------------------------
CREDIT_VARS_LEVEL = [
    "credit_market_level_log",
    "credit_curve_level"
]

CREDIT_VARS_LEVEL_PLUS_CHANGE = [
    "credit_market_level_log",
    "credit_curve_level",
    "credit_curve_change"
]

# -------------------------------------------------
# Validación: no obligues a que existan todas;
# validá cada familia por separado
# -------------------------------------------------
credit_blocks = {
    "legacy": CREDIT_VARS_LEGACY,
    "shock": CREDIT_VARS_SHOCK,
    "shock_l1": CREDIT_VARS_SHOCK_L1,
    "level": CREDIT_VARS_LEVEL,
    "level_plus_change": CREDIT_VARS_LEVEL_PLUS_CHANGE,
}

for block_name, block_vars in credit_blocks.items():
    missing_block = [v for v in block_vars if v not in df.columns]
    if missing_block:
        print(f"⚠ Bloque crédito '{block_name}' incompleto. Faltan: {missing_block}")
    else:
        print(f"✔ Bloque crédito '{block_name}' detectado")

if MARKET_SHOCK not in df.columns:
    raise ValueError(f"Falta variable clave del modelo: {MARKET_SHOCK}")

print("✔ Shock de mercado correctamente detectado")

# ==========================================================
# 3.6 MARKET POWER
# ==========================================================

# Proxy principal sugerida
MARKET_POWER = "market_share_industry_group"

# Set completo de proxies candidatas
MARKET_POWER_PROXIES = [
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
    "market_share_w",
    "market_share_raw",
]

available_market_power = [v for v in MARKET_POWER_PROXIES if v in df.columns]
missing_market_power = [v for v in MARKET_POWER_PROXIES if v not in df.columns]

print("\nProxies de market power detectadas en el panel:")
print(available_market_power)

if missing_market_power:
    print("\nProxies de market power no disponibles:")
    print(missing_market_power)

if MARKET_POWER not in df.columns:
    raise ValueError(f"Falta proxy principal de market power: {MARKET_POWER}")

print(f"✔ Proxy principal de market power correctamente detectada: {MARKET_POWER}")

# Debug útil: missingness y dispersión
print("\nResumen rápido de proxies de market power disponibles:")
for v in available_market_power:
    print(f"\n--- {v} ---")
    print("NaN share:", round(df[v].isna().mean(), 4))
    print("std:", round(df[v].std(skipna=True), 6))
    print("p95:", round(df[v].quantile(0.95), 6))
    print("max:", round(df[v].max(skipna=True), 6))

# ==========================================================
# 3.7 CHEQUEO FINAL DE VARIABLES DEL MODELO (SIN CRÉDITO RÍGIDO)
# ==========================================================

model_core_vars = (
    [DEP_VAR_MAIN, IVOL_VAR]
    + FIRM_CONTROLS
    + MACRO_VARS
    + [MARKET_SHOCK]
    + [MARKET_POWER]
)

missing_model_core = [v for v in model_core_vars if v not in df.columns]

if missing_model_core:
    raise ValueError(f"Faltan variables core del modelo: {missing_model_core}")

print("✔ Variables core (sin bloque de crédito) correctamente detectadas")


# -------------------------------------------------
# Check adicional: al menos un bloque de crédito válido
# -------------------------------------------------

valid_credit_blocks = []

for block_name, block_vars in credit_blocks.items():
    if all(v in df.columns for v in block_vars):
        valid_credit_blocks.append(block_name)

if len(valid_credit_blocks) == 0:
    raise ValueError("❌ No hay ningún bloque de crédito completo disponible")

print(f"✔ Bloques de crédito disponibles para modelar: {valid_credit_blocks}")

✔ Controles firm-level correctamente detectados
✔ Riesgo idiosincrático correctamente detectado
✔ Backbone macro-financiero correctamente detectado
✔ Bloque crédito 'legacy' detectado
✔ Bloque crédito 'shock' detectado
✔ Bloque crédito 'shock_l1' detectado
✔ Bloque crédito 'level' detectado
✔ Bloque crédito 'level_plus_change' detectado
✔ Shock de mercado correctamente detectado

Proxies de market power detectadas en el panel:
['market_share_sector', 'market_share_industry_group', 'market_share_industry', 'market_share_subindustry', 'market_share_w', 'market_share_raw']
✔ Proxy principal de market power correctamente detectada: market_share_industry_group

Resumen rápido de proxies de market power disponibles:

--- market_share_sector ---
NaN share: 0.0984
std: 0.048828
p95: 0.100759
max: 1.0

--- market_share_industry_group ---
NaN share: 0.0984
std: 0.095654
p95: 0.255283
max: 1.0

--- market_share_industry ---
NaN share: 0.0984
std: 0.184252
p95: 0.540911
max: 1.0

--- market_share_

## 4) Preparación de variables para estimación

En esta sección se prepara el panel para la estimación econométrica.

Se realizan los siguientes pasos:

1. Tratamiento de valores extremos
2. Manejo de valores faltantes
3. Validación de soporte de variables
4. Preparación del panel para modelos

---

## Principios econométricos

- No se imputan valores faltantes
- Los ratios se definen solo en su soporte válido
- Se controla la influencia de outliers
- Cada modelo utilizará su propia muestra efectiva (drop NA específico)

---

## Resultado esperado

Un panel limpio y consistente que puede ser utilizado para estimaciones con efectos fijos sin introducir sesgos mecánicos.

In [19]:

# ==========================================================
# 4.1 WINSORIZACIÓN
# ==========================================================

def winsorize_series(s, lower=0.01, upper=0.99):
    return s.clip(lower=s.quantile(lower), upper=s.quantile(upper))

# ==========================================================
# 4.1 WINSORIZACIÓN VARIABLES CLAVE
# ==========================================================


# Variables continuas sensibles
WINSOR_VARS = [
    "leverage",
    "cash_to_assets",
    "current_ratio",
    "interest_coverage",
    "rollover_12m",
    "residual_maturity_mean",
    "ivol_252",
    "oas_mean",
    "spread_mean_bps",
    "cds_5y_mean"
]

for var in WINSOR_VARS:
    if var in df.columns:
        df[var] = winsorize_series(df[var])

# ==========================================================
# 4.3 VALIDACIÓN DE SOPORTE
# ==========================================================

# eliminar valores absurdos
df.loc[df["leverage"] < 0, "leverage"] = np.nan

df.loc[df["current_ratio"] <= 0, "current_ratio"] = np.nan

df.loc[df["interest_coverage"] <= 0, "interest_coverage"] = np.nan

# ==========================================================
# 4.4 MISSING VALUES
# ==========================================================

# ------------------------------
# A) Variables core comunes
# ------------------------------
key_vars_core = (
    [DEP_VAR_MAIN]
    + FIRM_CONTROLS
    + MACRO_VARS
    + [IVOL_VAR]
    + [MARKET_SHOCK]
)

missing_core = df[key_vars_core].isna().mean().sort_values(ascending=False)

print("\n% missing variables core:")
print(missing_core)

# ------------------------------
# B) Missingness por bloques de crédito
# ------------------------------
print("\n" + "="*70)
print("MISSINGNESS POR BLOQUES DE CRÉDITO")
print("="*70)

credit_missing_tables = {}

for block_name, block_vars in credit_blocks.items():
    existing_vars = [v for v in block_vars if v in df.columns]

    if len(existing_vars) == 0:
        print(f"\n[{block_name}] Sin variables presentes en el panel")
        continue

    missing_block = df[existing_vars].isna().mean().sort_values(ascending=False)
    credit_missing_tables[block_name] = missing_block

    print(f"\n[{block_name}] % missing")
    print(missing_block)

# ------------------------------
# C) Check resumido: bloques completos disponibles
# ------------------------------
valid_credit_blocks = [
    block_name
    for block_name, block_vars in credit_blocks.items()
    if all(v in df.columns for v in block_vars)
]

print("\nBloques de crédito completos disponibles:")
print(valid_credit_blocks if valid_credit_blocks else "Ninguno")

# ==========================================================
# 4.5 INDEX PANEL
# ==========================================================

df = df.set_index(["issuer", "date"])

df = df.sort_index()

print("✔ Panel listo para estimación")

# ==========================================================
# 4.6 VARIACIÓN WITHIN
# ==========================================================

def check_within_variation(df, var):
    return df.groupby(level=0)[var].nunique().mean()

for var in ["leverage", "ivol_252", "mkt_ret"]:
    if var in df.columns:
        print(f"{var}: variación promedio dentro de firma =", check_within_variation(df, var))




% missing variables core:
rollover_12m              0.380649
leverage                  0.380405
current_ratio             0.343716
interest_coverage         0.300652
cash_to_assets            0.279746
log_assets                0.241141
ivol_252                  0.055678
oas_mean                  0.000000
residual_maturity_mean    0.000000
policy_rate               0.000000
term_spread_10y_2y        0.000000
move_eom                  0.000000
mkt_ret                   0.000000
dtype: float64

MISSINGNESS POR BLOQUES DE CRÉDITO

[legacy] % missing
credit_level    0.004425
credit_slope    0.004425
dtype: float64

[shock] % missing
credit_tightening_shock    0.004425
credit_curve_shock         0.004425
dtype: float64

[shock_l1] % missing
credit_tightening_shock_l1    0.00885
credit_curve_shock_l1         0.00885
dtype: float64

[level] % missing
credit_market_level_log    0.0
credit_curve_level         0.0
dtype: float64

[level_plus_change] % missing
credit_curve_change        0.004425


## 5) Construcción de interacciones

En esta sección se construyen variables de interacción para capturar heterogeneidad en la exposición de las firmas a shocks agregados.

---

## Motivación económica

Las firmas no responden homogéneamente a shocks de mercado o crédito. La exposición depende de características estructurales como:

- apalancamiento,
- riesgo de refinanciación,
- poder de mercado.

---

## Estrategia

Se construyen interacciones específicas y disciplinadas:

- Mercado:
  - mkt_ret × leverage
  - mkt_ret × market power

- Crédito:
  - credit_level × rollover
  - credit_level × market power

Estas interacciones permiten identificar mecanismos de transmisión heterogéneos.

In [20]:
# ==========================================================
# 5.1 INTERACCIONES — MERCADO
# ==========================================================

df["mkt_ret_x_leverage"] = df["mkt_ret"] * df["leverage"]

df["mkt_ret_x_market_power"] = df["mkt_ret"] * df[MARKET_POWER]

# ==========================================================
# 5.2 INTERACCIONES — CRÉDITO
# ==========================================================

df["credit_level_x_rollover"] = df["credit_level"] * df["rollover_12m"]

df["credit_level_x_market_power"] = df["credit_level"] * df[MARKET_POWER]

# ==========================================================
# 5.3 VALIDACIÓN
# ==========================================================

interaction_vars = [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power",
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]

missing_interactions = [v for v in interaction_vars if v not in df.columns]

if missing_interactions:
    raise ValueError(f"Faltan interacciones: {missing_interactions}")

print("✔ Interacciones construidas correctamente")

✔ Interacciones construidas correctamente


## 6) Definición de muestras

En esta sección se definen las muestras utilizadas para la estimación.

Se distinguen tres conjuntos de datos:

1. Muestra principal basada en OAS
2. Muestra alternativa basada en spread construido
3. Muestra alternativa basada en CDS

---

## Principio

Cada variable dependiente define su propia muestra.

No se utiliza una muestra común, ya que esto implicaría perder información innecesariamente.

---

## Objetivo

- maximizar uso de datos
- mantener consistencia dentro de cada especificación
- permitir comparaciones de robustez

In [21]:
# ==========================================================
# 6. PREPARACIÓN DE MUESTRAS E INTERACCIONES
# ==========================================================

# ==========================================================
# 6.1 FUNCIÓN DE MUESTRA
# ==========================================================

def get_sample(df, dep_var, required_vars):
    """
    Devuelve dataframe filtrado para estimación:
    - elimina NA solo en variables relevantes
    - evita duplicados accidentales en required_vars
    """
    cols_needed = [dep_var] + required_vars
    cols_needed = list(dict.fromkeys(cols_needed))  # preserva orden y elimina duplicados

    missing_cols = [c for c in cols_needed if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas para la muestra de {dep_var}: {missing_cols}")

    df_model = df[cols_needed].copy()
    df_model = df_model.dropna()

    return df_model


# ==========================================================
# 6.2 VARIABLES BASE COMUNES
# ==========================================================

BASE_VARS_COMMON = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS
    + [MARKET_SHOCK]
    + [MARKET_POWER]
)

# Bloques base por familia de crédito
BASE_VARS_BY_CREDIT = {
    "legacy": BASE_VARS_COMMON + CREDIT_VARS_LEGACY,
    "shock": BASE_VARS_COMMON + CREDIT_VARS_SHOCK,
    "shock_l1": BASE_VARS_COMMON + CREDIT_VARS_SHOCK_L1,
    "level": BASE_VARS_COMMON + CREDIT_VARS_LEVEL,
    "level_plus_change": BASE_VARS_COMMON + CREDIT_VARS_LEVEL_PLUS_CHANGE,
}


# ==========================================================
# 6.3 INTERACCIONES DE MERCADO
# ==========================================================

if {"mkt_ret", "leverage"}.issubset(df.columns):
    df["mkt_ret_x_leverage"] = df["mkt_ret"] * df["leverage"]

if {"mkt_ret", MARKET_POWER}.issubset(df.columns):
    df["mkt_ret_x_market_power"] = df["mkt_ret"] * df[MARKET_POWER]


# ==========================================================
# 6.4 INTERACCIONES DE CRÉDITO
# ==========================================================

# legacy
if {"credit_level", "rollover_12m"}.issubset(df.columns):
    df["credit_level_x_rollover"] = df["credit_level"] * df["rollover_12m"]

if {"credit_level", MARKET_POWER}.issubset(df.columns):
    df["credit_level_x_market_power"] = df["credit_level"] * df[MARKET_POWER]

# shocks
if {"credit_tightening_shock", "rollover_12m"}.issubset(df.columns):
    df["credit_tightening_shock_x_rollover"] = (
        df["credit_tightening_shock"] * df["rollover_12m"]
    )

if {"credit_tightening_shock", MARKET_POWER}.issubset(df.columns):
    df["credit_tightening_shock_x_market_power"] = (
        df["credit_tightening_shock"] * df[MARKET_POWER]
    )

# shocks laggeados
if {"credit_tightening_shock_l1", "rollover_12m"}.issubset(df.columns):
    df["credit_tightening_shock_l1_x_rollover"] = (
        df["credit_tightening_shock_l1"] * df["rollover_12m"]
    )

if {"credit_tightening_shock_l1", MARKET_POWER}.issubset(df.columns):
    df["credit_tightening_shock_l1_x_market_power"] = (
        df["credit_tightening_shock_l1"] * df[MARKET_POWER]
    )

# niveles
if {"credit_market_level_log", "rollover_12m"}.issubset(df.columns):
    df["credit_market_level_log_x_rollover"] = (
        df["credit_market_level_log"] * df["rollover_12m"]
    )

if {"credit_market_level_log", MARKET_POWER}.issubset(df.columns):
    df["credit_market_level_log_x_market_power"] = (
        df["credit_market_level_log"] * df[MARKET_POWER]
    )

print("✔ Interacciones de mercado y crédito construidas")


# ==========================================================
# 6.5 INTERACCIONES REQUERIDAS POR FAMILIA DE CRÉDITO
# ==========================================================

INTERACTION_VARS_BY_CREDIT = {
    "legacy": [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_level_x_rollover",
        "credit_level_x_market_power",
    ],
    "shock": [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_tightening_shock_x_rollover",
        "credit_tightening_shock_x_market_power",
    ],
    "shock_l1": [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_tightening_shock_l1_x_rollover",
        "credit_tightening_shock_l1_x_market_power",
    ],
    "level": [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_market_level_log_x_rollover",
        "credit_market_level_log_x_market_power",
    ],
    "level_plus_change": [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_market_level_log_x_rollover",
        "credit_market_level_log_x_market_power",
    ],
}


# ==========================================================
# 6.6 HELPER PARA ARMAR MUESTRAS POR DEPENDIENTE Y BLOQUE
# ==========================================================

def build_samples_for_depvar(df, dep_var, base_vars_by_credit, interaction_vars_by_credit):
    """
    Arma muestras por variable dependiente y por familia de crédito.
    """
    out = {}

    for block_name in base_vars_by_credit.keys():
        required_vars = base_vars_by_credit[block_name] + interaction_vars_by_credit[block_name]

        # solo usar columnas que realmente existan
        missing_vars = [v for v in required_vars if v not in df.columns]
        if missing_vars:
            print(f"⚠ {dep_var} | bloque '{block_name}' no disponible. Faltan: {missing_vars}")
            continue

        out[block_name] = get_sample(df, dep_var, required_vars)

    return out


# ==========================================================
# 6.7 MUESTRAS POR VARIABLE DEPENDIENTE
# ==========================================================

samples_oas = build_samples_for_depvar(
    df,
    DEP_VAR_MAIN,
    BASE_VARS_BY_CREDIT,
    INTERACTION_VARS_BY_CREDIT
)

samples_spread = build_samples_for_depvar(
    df,
    "spread_mean_bps",
    BASE_VARS_BY_CREDIT,
    INTERACTION_VARS_BY_CREDIT
)

samples_cds = build_samples_for_depvar(
    df,
    "cds_5y_mean",
    BASE_VARS_BY_CREDIT,
    INTERACTION_VARS_BY_CREDIT
)


# ==========================================================
# 6.8 ATAJOS COMPATIBLES CON EL NOTEBOOK ACTUAL
# ==========================================================

# Si querés mantener compatibilidad con el resto del notebook,
# definimos como default la familia legacy.
if "legacy" in samples_oas:
    df_oas = samples_oas["legacy"]
    print("\nMuestra OAS (legacy):", df_oas.shape)

if "legacy" in samples_spread:
    df_spread = samples_spread["legacy"]
    print("Muestra Spread (legacy):", df_spread.shape)

if "legacy" in samples_cds:
    df_cds = samples_cds["legacy"]
    print("Muestra CDS (legacy):", df_cds.shape)


# ==========================================================
# 6.9 COMPARACIÓN DE MUESTRAS
# ==========================================================

def sample_stats(df_sample, name):
    n_obs = len(df_sample)
    n_firms = df_sample.index.get_level_values(0).nunique()
    n_time = df_sample.index.get_level_values(1).nunique()

    print(f"\n{name}:")
    print(f"Observaciones: {n_obs}")
    print(f"Firmas: {n_firms}")
    print(f"Periodos: {n_time}")

if "legacy" in samples_oas:
    sample_stats(samples_oas["legacy"], "OAS | legacy")

if "legacy" in samples_spread:
    sample_stats(samples_spread["legacy"], "Spread | legacy")

if "legacy" in samples_cds:
    sample_stats(samples_cds["legacy"], "CDS | legacy")


# ==========================================================
# 6.10 RESUMEN DE DISPONIBILIDAD DE MUESTRAS
# ==========================================================

print("\n" + "="*70)
print("DISPONIBILIDAD DE MUESTRAS POR FAMILIA DE CRÉDITO")
print("="*70)

for dep_name, sample_dict in {
    "OAS": samples_oas,
    "Spread": samples_spread,
    "CDS": samples_cds
}.items():
    print(f"\n{dep_name}:")
    if len(sample_dict) == 0:
        print("  Ninguna muestra disponible")
        continue
    for block_name, df_sample in sample_dict.items():
        print(f"  - {block_name}: {df_sample.shape}")

✔ Interacciones de mercado y crédito construidas

Muestra OAS (legacy): (13969, 20)
Muestra Spread (legacy): (13969, 20)
Muestra CDS (legacy): (5470, 20)

OAS | legacy:
Observaciones: 13969
Firmas: 168
Periodos: 108

Spread | legacy:
Observaciones: 13969
Firmas: 168
Periodos: 108

CDS | legacy:
Observaciones: 5470
Firmas: 69
Periodos: 108

DISPONIBILIDAD DE MUESTRAS POR FAMILIA DE CRÉDITO

OAS:
  - legacy: (13969, 20)
  - shock: (13969, 20)
  - shock_l1: (13969, 20)
  - level: (13969, 20)
  - level_plus_change: (13969, 21)

Spread:
  - legacy: (13969, 20)
  - shock: (13969, 20)
  - shock_l1: (13969, 20)
  - level: (13969, 20)
  - level_plus_change: (13969, 21)

CDS:
  - legacy: (5470, 20)
  - shock: (5470, 20)
  - shock_l1: (5470, 20)
  - level: (5470, 20)
  - level_plus_change: (5470, 21)


## 7) Funciones de estimación

En esta sección se definen funciones auxiliares para estimar modelos de panel.

Las funciones permiten:

- estimar modelos con efectos fijos,
- controlar por efectos de tiempo cuando corresponde,
- clusterizar errores por firma,
- estructurar resultados de forma consistente.

---

## Especificación general

Los modelos se estiman mediante PanelOLS con:

- efectos fijos por firma en todos los modelos,
- efectos fijos de tiempo solo en el modelo benchmark (M0),
- errores estándar robustos clusterizados por firma.

In [22]:
# ==========================================================
# 7.1 FUNCIÓN BASE DE ESTIMACIÓN
# ==========================================================

from linearmodels.panel import PanelOLS

def run_panel_model(df, dep_var, exog_vars, add_time_fe=False):
    """
    Estima modelo PanelOLS con:
    - FE firma siempre
    - FE tiempo opcional
    - cluster por firma
    """
    
    y = df[dep_var]
    X = df[exog_vars]
    
    # Modelo
    model = PanelOLS(
        y,
        X,
        entity_effects=True,
        time_effects=add_time_fe
    )
    
    # Fit
    results = model.fit(
        cov_type="clustered",
        cluster_entity=True
    )
    
    return results

# ==========================================================
# 7.2 PREPARACIÓN DE DATASET POR MODELO
# ==========================================================

def prepare_model_data(df, dep_var, exog_vars):
    """
    Selecciona variables relevantes y elimina NA
    SOLO para ese modelo
    """
    
    cols = [dep_var] + exog_vars
    
    df_model = df[cols].dropna()
    
    return df_model

# ==========================================================
# 7.3 FUNCIÓN COMPLETA DE ESTIMACIÓN
# ==========================================================

def estimate_model(df, dep_var, exog_vars, add_time_fe=False):
    
    df_model = prepare_model_data(df, dep_var, exog_vars)
    
    results = run_panel_model(
        df_model,
        dep_var,
        exog_vars,
        add_time_fe=add_time_fe
    )
    
    return results, df_model

# ==========================================================
# 7.4 RESUMEN DE MUESTRA
# ==========================================================

def sample_info(df_model):
    
    n_obs = len(df_model)
    n_firms = df_model.index.get_level_values(0).nunique()
    n_time = df_model.index.get_level_values(1).nunique()
    
    return {
        "n_obs": n_obs,
        "n_firms": n_firms,
        "n_time": n_time
    }

# ==========================================================
# 7.5 TEST RÁPIDO
# ==========================================================

test_vars = [
    "leverage",
    "log_assets",
    "mkt_ret"
]

res_test, df_test = estimate_model(
    df,
    DEP_VAR_MAIN,
    test_vars,
    add_time_fe=False
)

print(res_test.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:               oas_mean   R-squared:                        0.0603
Estimator:                   PanelOLS   R-squared (Between):             -49.130
No. Observations:               17783   R-squared (Within):               0.0603
Date:                Thu, Mar 26 2026   R-squared (Overall):             -43.629
Time:                        15:29:10   Log-likelihood                -9.279e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      376.14
Entities:                         197   P-value                           0.0000
Avg Obs:                       90.269   Distribution:                 F(3,17583)
Min Obs:                       5.0000                                           
Max Obs:                       109.00   F-statistic (robust):             192.33
                            

## 8) Definición de modelos M0–M6

En esta sección se definen las especificaciones empíricas utilizadas en el análisis.

La secuencia de modelos permite identificar progresivamente:

1. un benchmark con efectos fijos completos,
2. un backbone macro-financiero observable,
3. el efecto de shocks de mercado,
4. heterogeneidad en la exposición a shocks de mercado,
5. el efecto de shocks de crédito,
6. heterogeneidad en la exposición a shocks de crédito,
7. un modelo integrado final.

---

## Estructura

- M0: Benchmark (FE firma + FE tiempo)
- M1: Backbone macro-financiero
- M2: Shock de mercado
- M3: Mercado heterogéneo
- M4: Shock de crédito
- M5: Crédito heterogéneo
- M6: Modelo integrado

In [23]:
# ==========================================================
# MACRO VARS
# ==========================================================

MACRO_VARS_BASE = MACRO_VARS.copy()
MACRO_VARS_COVID = MACRO_VARS_BASE + ["covid_dummy"]

# ==========================================================
# BLOQUES DE CRÉDITO
# ==========================================================

# Legacy (backward compatible)
CREDIT_VARS_LEGACY = [
    "credit_level",
    "credit_slope"
]

# Nuevos nombres de shocks
CREDIT_VARS_SHOCK = [
    "credit_tightening_shock",
    "credit_curve_shock"
]

# Shocks laggeados
CREDIT_VARS_LAG = [
    "credit_tightening_shock_l1",
    "credit_curve_shock_l1"
]

# Alternativas en nivel
CREDIT_VARS_LEVEL = [
    "credit_market_level_log",
    "credit_curve_level",
    "credit_curve_change"
]

# ==========================================================
# M0 — BENCHMARK
# ==========================================================

M0_VARS = FIRM_CONTROLS + [IVOL_VAR]

M0_CONFIG = {
    "vars": M0_VARS,
    "time_fe": True
}

# ==========================================================
# M1 — BACKBONE MACRO
# ==========================================================

M1_VARS = FIRM_CONTROLS + [IVOL_VAR] + MACRO_VARS_BASE

M1_CONFIG = {
    "vars": M1_VARS,
    "time_fe": False
}

# ==========================================================
# M2 — SHOCK DE MERCADO
# ==========================================================

M2_VARS = M1_VARS + [MARKET_SHOCK]

M2_CONFIG = {
    "vars": M2_VARS,
    "time_fe": False
}

# ==========================================================
# M3 — MERCADO HETEROGÉNEO
# ==========================================================

M3_VARS = M2_VARS + [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power"
]

M3_CONFIG = {
    "vars": M3_VARS,
    "time_fe": False
}

# ==========================================================
# M4–M6 PRINCIPALES (LEGACY)
# ==========================================================

M4_VARS = M1_VARS + CREDIT_VARS_LEGACY

M4_CONFIG = {
    "vars": M4_VARS,
    "time_fe": False
}

M5_VARS = M4_VARS + [
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]

M5_CONFIG = {
    "vars": M5_VARS,
    "time_fe": False
}

M6_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_BASE
    + [MARKET_SHOCK]
    + CREDIT_VARS_LEGACY
    + [
        "mkt_ret_x_leverage",
        "credit_level_x_rollover"
    ]
)

M6_CONFIG = {
    "vars": M6_VARS,
    "time_fe": False
}

# ==========================================================
# M4–M6 ALTERNATIVA 1: SHOCKS RENOMBRADOS
# ==========================================================

M4_SHOCK_VARS = M1_VARS + CREDIT_VARS_SHOCK

M4_SHOCK_CONFIG = {
    "vars": M4_SHOCK_VARS,
    "time_fe": False
}

M5_SHOCK_VARS = M4_SHOCK_VARS + [
    "credit_tightening_shock_x_rollover",
    "credit_tightening_shock_x_market_power"
]

M5_SHOCK_CONFIG = {
    "vars": M5_SHOCK_VARS,
    "time_fe": False
}

M6_SHOCK_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_BASE
    + [MARKET_SHOCK]
    + CREDIT_VARS_SHOCK
    + [
        "mkt_ret_x_leverage",
        "credit_tightening_shock_x_rollover"
    ]
)

M6_SHOCK_CONFIG = {
    "vars": M6_SHOCK_VARS,
    "time_fe": False
}

# ==========================================================
# M4–M6 ALTERNATIVA 2: SHOCKS LAGGEADOS
# ==========================================================

M4_LAG_VARS = M1_VARS + CREDIT_VARS_LAG

M4_LAG_CONFIG = {
    "vars": M4_LAG_VARS,
    "time_fe": False
}

M5_LAG_VARS = M4_LAG_VARS + [
    "credit_tightening_shock_l1_x_rollover",
    "credit_tightening_shock_l1_x_market_power"
]

M5_LAG_CONFIG = {
    "vars": M5_LAG_VARS,
    "time_fe": False
}

M6_LAG_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_BASE
    + [MARKET_SHOCK]
    + CREDIT_VARS_LAG
    + [
        "mkt_ret_x_leverage",
        "credit_tightening_shock_l1_x_rollover"
    ]
)

M6_LAG_CONFIG = {
    "vars": M6_LAG_VARS,
    "time_fe": False
}

# ==========================================================
# M4–M6 ALTERNATIVA 3: NIVELES
# ==========================================================

M4_LEVEL_VARS = M1_VARS + CREDIT_VARS_LEVEL

M4_LEVEL_CONFIG = {
    "vars": M4_LEVEL_VARS,
    "time_fe": False
}

M5_LEVEL_VARS = M4_LEVEL_VARS + [
    "credit_market_level_log_x_rollover",
    "credit_market_level_log_x_market_power"
]

M5_LEVEL_CONFIG = {
    "vars": M5_LEVEL_VARS,
    "time_fe": False
}

M6_LEVEL_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_BASE
    + [MARKET_SHOCK]
    + CREDIT_VARS_LEVEL
    + [
        "mkt_ret_x_leverage",
        "credit_market_level_log_x_rollover"
    ]
)

M6_LEVEL_CONFIG = {
    "vars": M6_LEVEL_VARS,
    "time_fe": False
}

# ==========================================================
# MODELOS + COVID DUMMY
# ==========================================================

# ------------------------------
# M1–M3 con COVID
# ------------------------------
M1_COVID_VARS = FIRM_CONTROLS + [IVOL_VAR] + MACRO_VARS_COVID
M2_COVID_VARS = M1_COVID_VARS + [MARKET_SHOCK]
M3_COVID_VARS = M2_COVID_VARS + [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power"
]

M1_COVID_CONFIG = {"vars": M1_COVID_VARS, "time_fe": False}
M2_COVID_CONFIG = {"vars": M2_COVID_VARS, "time_fe": False}
M3_COVID_CONFIG = {"vars": M3_COVID_VARS, "time_fe": False}

# ------------------------------
# LEGACY + COVID
# ------------------------------
M4_COVID_VARS = M1_COVID_VARS + CREDIT_VARS_LEGACY
M5_COVID_VARS = M4_COVID_VARS + [
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]
M6_COVID_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_COVID
    + [MARKET_SHOCK]
    + CREDIT_VARS_LEGACY
    + [
        "mkt_ret_x_leverage",
        "credit_level_x_rollover"
    ]
)

M4_COVID_CONFIG = {"vars": M4_COVID_VARS, "time_fe": False}
M5_COVID_CONFIG = {"vars": M5_COVID_VARS, "time_fe": False}
M6_COVID_CONFIG = {"vars": M6_COVID_VARS, "time_fe": False}

# ------------------------------
# SHOCK + COVID
# ------------------------------
M4_SHOCK_COVID_VARS = M1_COVID_VARS + CREDIT_VARS_SHOCK
M5_SHOCK_COVID_VARS = M4_SHOCK_COVID_VARS + [
    "credit_tightening_shock_x_rollover",
    "credit_tightening_shock_x_market_power"
]
M6_SHOCK_COVID_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_COVID
    + [MARKET_SHOCK]
    + CREDIT_VARS_SHOCK
    + [
        "mkt_ret_x_leverage",
        "credit_tightening_shock_x_rollover"
    ]
)

M4_SHOCK_COVID_CONFIG = {"vars": M4_SHOCK_COVID_VARS, "time_fe": False}
M5_SHOCK_COVID_CONFIG = {"vars": M5_SHOCK_COVID_VARS, "time_fe": False}
M6_SHOCK_COVID_CONFIG = {"vars": M6_SHOCK_COVID_VARS, "time_fe": False}

# ------------------------------
# LAG + COVID
# ------------------------------
M4_LAG_COVID_VARS = M1_COVID_VARS + CREDIT_VARS_LAG
M5_LAG_COVID_VARS = M4_LAG_COVID_VARS + [
    "credit_tightening_shock_l1_x_rollover",
    "credit_tightening_shock_l1_x_market_power"
]
M6_LAG_COVID_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_COVID
    + [MARKET_SHOCK]
    + CREDIT_VARS_LAG
    + [
        "mkt_ret_x_leverage",
        "credit_tightening_shock_l1_x_rollover"
    ]
)

M4_LAG_COVID_CONFIG = {"vars": M4_LAG_COVID_VARS, "time_fe": False}
M5_LAG_COVID_CONFIG = {"vars": M5_LAG_COVID_VARS, "time_fe": False}
M6_LAG_COVID_CONFIG = {"vars": M6_LAG_COVID_VARS, "time_fe": False}

# ------------------------------
# LEVEL + COVID
# ------------------------------
M4_LEVEL_COVID_VARS = M1_COVID_VARS + CREDIT_VARS_LEVEL
M5_LEVEL_COVID_VARS = M4_LEVEL_COVID_VARS + [
    "credit_market_level_log_x_rollover",
    "credit_market_level_log_x_market_power"
]
M6_LEVEL_COVID_VARS = (
    FIRM_CONTROLS
    + [IVOL_VAR]
    + MACRO_VARS_COVID
    + [MARKET_SHOCK]
    + CREDIT_VARS_LEVEL
    + [
        "mkt_ret_x_leverage",
        "credit_market_level_log_x_rollover"
    ]
)

M4_LEVEL_COVID_CONFIG = {"vars": M4_LEVEL_COVID_VARS, "time_fe": False}
M5_LEVEL_COVID_CONFIG = {"vars": M5_LEVEL_COVID_VARS, "time_fe": False}
M6_LEVEL_COVID_CONFIG = {"vars": M6_LEVEL_COVID_VARS, "time_fe": False}

# ==========================================================
# MODELOS BASE
# ==========================================================

MODEL_SPECS_BASE = {
    "M0": M0_CONFIG,
    "M1": M1_CONFIG,
    "M2": M2_CONFIG,
    "M3": M3_CONFIG,

    "M4": M4_CONFIG,
    "M5": M5_CONFIG,
    "M6": M6_CONFIG,

    "M4_shock": M4_SHOCK_CONFIG,
    "M5_shock": M5_SHOCK_CONFIG,
    "M6_shock": M6_SHOCK_CONFIG,

    "M4_lag": M4_LAG_CONFIG,
    "M5_lag": M5_LAG_CONFIG,
    "M6_lag": M6_LAG_CONFIG,

    "M4_level": M4_LEVEL_CONFIG,
    "M5_level": M5_LEVEL_CONFIG,
    "M6_level": M6_LEVEL_CONFIG,
}

# ==========================================================
# MODELOS + COVID
# ==========================================================

MODEL_SPECS_COVID = {
    "M0": M0_CONFIG,  # time FE absorbe covid_dummy

    "M1_covid": M1_COVID_CONFIG,
    "M2_covid": M2_COVID_CONFIG,
    "M3_covid": M3_COVID_CONFIG,

    "M4_covid": M4_COVID_CONFIG,
    "M5_covid": M5_COVID_CONFIG,
    "M6_covid": M6_COVID_CONFIG,

    "M4_shock_covid": M4_SHOCK_COVID_CONFIG,
    "M5_shock_covid": M5_SHOCK_COVID_CONFIG,
    "M6_shock_covid": M6_SHOCK_COVID_CONFIG,

    "M4_lag_covid": M4_LAG_COVID_CONFIG,
    "M5_lag_covid": M5_LAG_COVID_CONFIG,
    "M6_lag_covid": M6_LAG_COVID_CONFIG,

    "M4_level_covid": M4_LEVEL_COVID_CONFIG,
    "M5_level_covid": M5_LEVEL_COVID_CONFIG,
    "M6_level_covid": M6_LEVEL_COVID_CONFIG,
}

# ==========================================================
# 9. ESTIMACIÓN DE MODELOS Y GENERACIÓN DE OUTPUTS
# ==========================================================

## 🎯 Objetivo del bloque

Este bloque implementa la **estimación sistemática de todos los modelos econométricos** definidos en la tesis, organizados en distintas **baterías de especificaciones**.

El objetivo es:

- Ejecutar de forma automatizada múltiples modelos (M0–M6 y variantes)
- Mantener consistencia en la estimación
- Guardar resultados en formatos exportables
- Facilitar la comparación entre especificaciones

---

## ⚙️ Estructura general

El bloque se divide en cuatro partes:

### 1️⃣ Función principal de estimación (`run_model_battery`)

Esta función ejecuta una **batería completa de modelos** sobre el mismo dataset.

Para cada modelo:

- Estima la regresión usando `estimate_model()`
- Construye información de muestra (`sample_info`)
- Guarda metadatos de la especificación
- Exporta outputs

#### Outputs generados:

📄 **Summary del modelo**  
→ `.txt` con tabla completa de resultados (coeficientes, errores estándar, etc.)

📊 **Muestra utilizada**  
→ `.parquet` con el dataset efectivamente usado en la estimación

Esto es clave para:

- reproducibilidad
- debugging
- validación de resultados

---

### 2️⃣ Definición de baterías de modelos

Se agrupan las especificaciones en distintas baterías según su propósito empírico:

#### 🟦 Baterías BASE

- `main` → modelos principales (M0–M6)
- `shock` → versiones con shocks contemporáneos
- `lag` → versiones con shocks rezagados
- `level` → especificaciones en niveles

#### 🟥 Baterías COVID

Replican las anteriores pero incorporando:

- dummy COVID
- interacciones con el período de pandemia

---

### 3️⃣ Estimación de todas las baterías

Se ejecuta `run_model_battery()` para cada conjunto:

```python
main, shock, lag, level
main_covid, shock_covid, lag_covid, level_covid

In [24]:
# ==========================================================
# 9.1 FUNCIÓN DE ESTIMACIÓN COMPLETA (BATERÍA)
# ==========================================================

def run_model_battery(df, dep_var, model_specs, battery_name):

    results_dict = {}
    sample_dict = {}
    spec_dict = {}

    # Subcarpetas específicas por batería
    battery_summary_dir = MODEL_SUMMARY_DIR / battery_name
    battery_sample_dir = MODEL_SAMPLE_DIR / battery_name

    battery_summary_dir.mkdir(parents=True, exist_ok=True)
    battery_sample_dir.mkdir(parents=True, exist_ok=True)

    for model_name, config in model_specs.items():

        print(f"\n==============================")
        print(f"{battery_name.upper()} | Estimando {model_name}")
        print(f"==============================")

        vars_model = config["vars"]
        time_fe = config["time_fe"]

        res, df_model = estimate_model(
            df,
            dep_var,
            vars_model,
            add_time_fe=time_fe
        )

        results_dict[model_name] = res
        sample_dict[model_name] = sample_info(df_model)
        spec_dict[model_name] = {
            "dep_var": dep_var,
            "rhs_vars": " | ".join(vars_model),
            "time_fe": time_fe,
            "firm_fe": True,
            "battery": battery_name,
        }

        print(f"Observaciones: {sample_dict[model_name]['n_obs']}")
        print(f"Firmas: {sample_dict[model_name]['n_firms']}")
        print(f"Periodos: {sample_dict[model_name]['n_time']}")

        # Guardar summary
        with open(battery_summary_dir / f"{model_name}_{dep_var}_summary.txt", "w", encoding="utf-8") as f:
            f.write(str(res.summary))

        # Guardar muestra
        df_model.to_parquet(
            battery_sample_dir / f"{model_name}_{dep_var}_sample.parquet",
            index=False
        )

    return results_dict, sample_dict, spec_dict


# ==========================================================
# 9.2 DEFINICIÓN DE BATERÍAS
# ==========================================================

# ------------------------------
# BASE
# ------------------------------
MODEL_SPECS_MAIN = {
    k: v for k, v in MODEL_SPECS_BASE.items()
    if k in ["M0", "M1", "M2", "M3", "M4", "M5", "M6"]
}

MODEL_SPECS_SHOCK = {
    k: v for k, v in MODEL_SPECS_BASE.items()
    if k in ["M4_shock", "M5_shock", "M6_shock"]
}

MODEL_SPECS_LAG = {
    k: v for k, v in MODEL_SPECS_BASE.items()
    if k in ["M4_lag", "M5_lag", "M6_lag"]
}

MODEL_SPECS_LEVEL = {
    k: v for k, v in MODEL_SPECS_BASE.items()
    if k in ["M4_level", "M5_level", "M6_level"]
}

# ------------------------------
# COVID
# ------------------------------
MODEL_SPECS_MAIN_COVID = {
    k: v for k, v in MODEL_SPECS_COVID.items()
    if k in ["M0", "M1_covid", "M2_covid", "M3_covid", "M4_covid", "M5_covid", "M6_covid"]
}

MODEL_SPECS_SHOCK_COVID = {
    k: v for k, v in MODEL_SPECS_COVID.items()
    if k in ["M4_shock_covid", "M5_shock_covid", "M6_shock_covid"]
}

MODEL_SPECS_LAG_COVID = {
    k: v for k, v in MODEL_SPECS_COVID.items()
    if k in ["M4_lag_covid", "M5_lag_covid", "M6_lag_covid"]
}

MODEL_SPECS_LEVEL_COVID = {
    k: v for k, v in MODEL_SPECS_COVID.items()
    if k in ["M4_level_covid", "M5_level_covid", "M6_level_covid"]
}


# ==========================================================
# 9.3 ESTIMACIÓN DE MODELOS (BATERÍAS)
# ==========================================================

MODEL_SUMMARY_DIR = RESULTS_DIR / "model_summaries"
MODEL_SAMPLE_DIR = RESULTS_DIR / "model_samples"

MODEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# BASE
# ------------------------------
results_main, sample_main, spec_main = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_MAIN,
    battery_name="main"
)

results_shock, sample_shock, spec_shock = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_SHOCK,
    battery_name="shock"
)

results_lag, sample_lag, spec_lag = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_LAG,
    battery_name="lag"
)

results_level, sample_level, spec_level = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_LEVEL,
    battery_name="level"
)

# ------------------------------
# COVID
# ------------------------------
results_main_covid, sample_main_covid, spec_main_covid = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_MAIN_COVID,
    battery_name="main_covid"
)

results_shock_covid, sample_shock_covid, spec_shock_covid = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_SHOCK_COVID,
    battery_name="shock_covid"
)

results_lag_covid, sample_lag_covid, spec_lag_covid = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_LAG_COVID,
    battery_name="lag_covid"
)

results_level_covid, sample_level_covid, spec_level_covid = run_model_battery(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_LEVEL_COVID,
    battery_name="level_covid"
)


# ==========================================================
# 9.4 FUNCIÓN PARA GENERAR PAPER TABLE
# ==========================================================

def build_paper_table(results_dict, sample_dict, dep_var, suffix):

    def stars(p):
        if pd.isna(p):
            return ""
        if p < 0.01:
            return "***"
        elif p < 0.05:
            return "**"
        elif p < 0.10:
            return "*"
        return ""

    paper_rows = []

    # lista de variables únicas
    all_vars = []
    for model_name, res in results_dict.items():
        for v in res.params.index:
            if v not in all_vars:
                all_vars.append(v)

    # opcional: orden alfabético para prolijidad
    all_vars = sorted(all_vars)

    # coeficientes + SE
    for var in all_vars:
        row_coef = {"variable": var}
        row_se = {"variable": ""}

        for model_name, res in results_dict.items():
            if var in res.params.index:
                coef = res.params[var]
                se = res.std_errors[var]
                p = res.pvalues[var]

                row_coef[model_name] = f"{coef:.4f}{stars(p)}"
                row_se[model_name] = f"({se:.4f})"
            else:
                row_coef[model_name] = ""
                row_se[model_name] = ""

        paper_rows.append(row_coef)
        paper_rows.append(row_se)

    # stats al pie
    stats_labels = {
        "n_obs": "Observaciones",
        "n_firms": "Firmas",
        "n_time": "Periodos",
    }

    for key, label in stats_labels.items():
        row = {"variable": label}
        for model_name in results_dict.keys():
            row[model_name] = sample_dict[model_name][key]
        paper_rows.append(row)

    paper_table = pd.DataFrame(paper_rows)

    paper_table.to_csv(TABLES_DIR / f"paper_table_{dep_var}_{suffix}.csv", index=False)
    paper_table.to_excel(TABLES_DIR / f"paper_table_{dep_var}_{suffix}.xlsx", index=False)

    print(f"✅ Paper table ({suffix}) guardada")

    return paper_table


# ==========================================================
# 9.5 GENERAR TABLAS PAPER
# ==========================================================

# ------------------------------
# BASE
# ------------------------------
paper_main = build_paper_table(
    results_main,
    sample_main,
    DEP_VAR_MAIN,
    suffix="main"
)

paper_shock = build_paper_table(
    results_shock,
    sample_shock,
    DEP_VAR_MAIN,
    suffix="shock"
)

paper_lag = build_paper_table(
    results_lag,
    sample_lag,
    DEP_VAR_MAIN,
    suffix="lag"
)

paper_level = build_paper_table(
    results_level,
    sample_level,
    DEP_VAR_MAIN,
    suffix="level"
)

# ------------------------------
# COVID
# ------------------------------
paper_main_covid = build_paper_table(
    results_main_covid,
    sample_main_covid,
    DEP_VAR_MAIN,
    suffix="main_covid"
)

paper_shock_covid = build_paper_table(
    results_shock_covid,
    sample_shock_covid,
    DEP_VAR_MAIN,
    suffix="shock_covid"
)

paper_lag_covid = build_paper_table(
    results_lag_covid,
    sample_lag_covid,
    DEP_VAR_MAIN,
    suffix="lag_covid"
)

paper_level_covid = build_paper_table(
    results_level_covid,
    sample_level_covid,
    DEP_VAR_MAIN,
    suffix="level_covid"
)


# ==========================================================
# 9.6 SAMPLE SUMMARY
# ==========================================================

def build_sample_summary(sample_dict, spec_dict, dep_var, suffix):

    sample_df = pd.DataFrame(sample_dict).T
    spec_df = pd.DataFrame(spec_dict).T
    sample_summary = sample_df.join(spec_df)

    sample_summary.to_csv(TABLES_DIR / f"sample_summary_{dep_var}_{suffix}.csv", index=True)
    sample_summary.to_excel(TABLES_DIR / f"sample_summary_{dep_var}_{suffix}.xlsx", index=True)

    print(f"✅ Sample summary ({suffix}) guardado")

    return sample_summary

# ------------------------------
# BASE
# ------------------------------
sample_summary_main = build_sample_summary(sample_main, spec_main, DEP_VAR_MAIN, "main")
sample_summary_shock = build_sample_summary(sample_shock, spec_shock, DEP_VAR_MAIN, "shock")
sample_summary_lag = build_sample_summary(sample_lag, spec_lag, DEP_VAR_MAIN, "lag")
sample_summary_level = build_sample_summary(sample_level, spec_level, DEP_VAR_MAIN, "level")

# ------------------------------
# COVID
# ------------------------------
sample_summary_main_covid = build_sample_summary(sample_main_covid, spec_main_covid, DEP_VAR_MAIN, "main_covid")
sample_summary_shock_covid = build_sample_summary(sample_shock_covid, spec_shock_covid, DEP_VAR_MAIN, "shock_covid")
sample_summary_lag_covid = build_sample_summary(sample_lag_covid, spec_lag_covid, DEP_VAR_MAIN, "lag_covid")
sample_summary_level_covid = build_sample_summary(sample_level_covid, spec_level_covid, DEP_VAR_MAIN, "level_covid")


# ==========================================================
# 9.7 METADATA DE CORRIDA
# ==========================================================

def build_run_metadata(sample_dict, spec_dict, dep_var, suffix):

    run_meta = pd.DataFrame({
        "model": list(sample_dict.keys()),
        "dep_var": [dep_var] * len(sample_dict),
        "time_fe": [spec_dict[m]["time_fe"] for m in sample_dict.keys()],
        "rhs_vars": [spec_dict[m]["rhs_vars"] for m in sample_dict.keys()],
        "battery": [spec_dict[m]["battery"] for m in sample_dict.keys()],
        "n_obs": [sample_dict[m]["n_obs"] for m in sample_dict.keys()],
        "n_firms": [sample_dict[m]["n_firms"] for m in sample_dict.keys()],
        "n_time": [sample_dict[m]["n_time"] for m in sample_dict.keys()],
    })

    run_meta.to_csv(RESULTS_DIR / f"run_metadata_{dep_var}_{suffix}.csv", index=False)
    run_meta.to_excel(RESULTS_DIR / f"run_metadata_{dep_var}_{suffix}.xlsx", index=False)

    print(f"✅ Metadata ({suffix}) guardada")

    return run_meta

# ------------------------------
# BASE
# ------------------------------
run_meta_main = build_run_metadata(sample_main, spec_main, DEP_VAR_MAIN, "main")
run_meta_shock = build_run_metadata(sample_shock, spec_shock, DEP_VAR_MAIN, "shock")
run_meta_lag = build_run_metadata(sample_lag, spec_lag, DEP_VAR_MAIN, "lag")
run_meta_level = build_run_metadata(sample_level, spec_level, DEP_VAR_MAIN, "level")

# ------------------------------
# COVID
# ------------------------------
run_meta_main_covid = build_run_metadata(sample_main_covid, spec_main_covid, DEP_VAR_MAIN, "main_covid")
run_meta_shock_covid = build_run_metadata(sample_shock_covid, spec_shock_covid, DEP_VAR_MAIN, "shock_covid")
run_meta_lag_covid = build_run_metadata(sample_lag_covid, spec_lag_covid, DEP_VAR_MAIN, "lag_covid")
run_meta_level_covid = build_run_metadata(sample_level_covid, spec_level_covid, DEP_VAR_MAIN, "level_covid")


# ==========================================================
# 9.8 LOG FINAL
# ==========================================================

print("\n=== ARCHIVOS EXPORTADOS ===")

print("\nTables:")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_main.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_shock.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_lag.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_level.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_main_covid.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_shock_covid.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_lag_covid.csv")
print(TABLES_DIR / f"paper_table_{DEP_VAR_MAIN}_level_covid.csv")

print("\nMetadata:")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_main.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_shock.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_lag.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_level.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_main_covid.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_shock_covid.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_lag_covid.csv")
print(RESULTS_DIR / f"run_metadata_{DEP_VAR_MAIN}_level_covid.csv")

print("\nModel summaries:")
print(MODEL_SUMMARY_DIR / "main")
print(MODEL_SUMMARY_DIR / "shock")
print(MODEL_SUMMARY_DIR / "lag")
print(MODEL_SUMMARY_DIR / "level")
print(MODEL_SUMMARY_DIR / "main_covid")
print(MODEL_SUMMARY_DIR / "shock_covid")
print(MODEL_SUMMARY_DIR / "lag_covid")
print(MODEL_SUMMARY_DIR / "level_covid")

print("\nModel samples:")
print(MODEL_SAMPLE_DIR / "main")
print(MODEL_SAMPLE_DIR / "shock")
print(MODEL_SAMPLE_DIR / "lag")
print(MODEL_SAMPLE_DIR / "level")
print(MODEL_SAMPLE_DIR / "main_covid")
print(MODEL_SAMPLE_DIR / "shock_covid")
print(MODEL_SAMPLE_DIR / "lag_covid")
print(MODEL_SAMPLE_DIR / "level_covid")


MAIN | Estimando M0
Observaciones: 15706
Firmas: 170
Periodos: 109

MAIN | Estimando M1
Observaciones: 15706
Firmas: 170
Periodos: 109

MAIN | Estimando M2
Observaciones: 15706
Firmas: 170
Periodos: 109

MAIN | Estimando M3
Observaciones: 13969
Firmas: 168
Periodos: 108

MAIN | Estimando M4
Observaciones: 15706
Firmas: 170
Periodos: 109

MAIN | Estimando M5
Observaciones: 13969
Firmas: 168
Periodos: 108

MAIN | Estimando M6
Observaciones: 15706
Firmas: 170
Periodos: 109

SHOCK | Estimando M4_shock
Observaciones: 15706
Firmas: 170
Periodos: 109

SHOCK | Estimando M5_shock
Observaciones: 13969
Firmas: 168
Periodos: 108

SHOCK | Estimando M6_shock
Observaciones: 15706
Firmas: 170
Periodos: 109

LAG | Estimando M4_lag
Observaciones: 15706
Firmas: 170
Periodos: 109

LAG | Estimando M5_lag
Observaciones: 13969
Firmas: 168
Periodos: 108

LAG | Estimando M6_lag
Observaciones: 15706
Firmas: 170
Periodos: 109

LEVEL | Estimando M4_level
Observaciones: 15706
Firmas: 170
Periodos: 109

LEVEL | Es

## 10) Robustez temporal — M6 por submuestras

En esta sección se evalúa la estabilidad temporal del modelo final **M6** mediante una partición del panel en tres submuestras asociadas a distintos regímenes macro-financieros:

- **pre_covid**: hasta 2019
- **covid**: 2020–2021
- **post_covid**: 2022 en adelante

---

## Objetivo econométrico

El objetivo es verificar si la relación entre los credit spreads y los shocks agregados de mercado y crédito, junto con la heterogeneidad firm-level, se mantiene estable a lo largo del tiempo o si presenta evidencia de cambio estructural.

La lógica del ejercicio es estricta:

- se mantiene **exactamente la misma especificación de M6**,
- se modifica **solo la muestra temporal**,
- no se agregan ni eliminan regresores,
- no se cambia la estructura de efectos fijos ni el tratamiento de errores.

De este modo, cualquier diferencia en coeficientes, signos o significancia puede interpretarse como evidencia de variación entre regímenes y no como consecuencia de una redefinición del modelo.

---

## Decisión metodológica importante

La muestra válida se recalcula dentro de cada submuestra.

Esto implica que para cada período:

1. primero se filtra el panel por rango temporal,
2. luego se aplica el `dropna()` solo sobre las variables requeridas por M6.

Este criterio evita imponer artificialmente restricciones de missing values de un período sobre otro y asegura que cada regresión utilice su muestra efectiva propia.

---

## Outputs generados

Esta sección produce:

- estimaciones de **M6_pre**, **M6_covid** y **M6_post**,
- archivos `.txt` con los summaries de cada regresión,
- muestras efectivas guardadas en `.parquet`,
- una **paper table consolidada** con las tres columnas comparables,
- tablas de resumen muestral y metadata de corrida.

---

## Criterio de lectura

La comparación entre submuestras debe enfocarse en:

1. estabilidad de signos,
2. cambios en magnitudes,
3. pérdida o ganancia de significancia,
4. variaciones en tamaño muestral, número de firmas y número de períodos.

Una mayor sensibilidad en la submuestra COVID puede interpretarse como intensificación de la transmisión de shocks agregados en un régimen de estrés. En cambio, cambios de signo o desaparición sistemática de variables clave pueden sugerir fragilidad de identificación o dependencia de régimen.

In [25]:
# ==========================================================
# 10. ROBUSTEZ TEMPORAL — M6 POR SUBMUESTRAS
# ==========================================================

# ==========================================================
# 10.1 CONFIGURACIÓN DEL MODELO FINAL
# ==========================================================

# Modelo final a testear por submuestras
FINAL_MODEL_NAME = "M6"
FINAL_MODEL_CONFIG = M6_CONFIG.copy()

FINAL_DEP_VAR = DEP_VAR_MAIN
FINAL_VARS = FINAL_MODEL_CONFIG["vars"]
FINAL_TIME_FE = FINAL_MODEL_CONFIG["time_fe"]

print("="*70)
print("ROBUSTEZ TEMPORAL — MODELO FINAL")
print("="*70)
print("Modelo:", FINAL_MODEL_NAME)
print("Variable dependiente:", FINAL_DEP_VAR)
print("Time FE:", FINAL_TIME_FE)
print("N regressors:", len(FINAL_VARS))
print("Regresores:")
for v in FINAL_VARS:
    print(" -", v)


# ==========================================================
# 10.2 DEFINICIÓN DE SUBMUESTRAS
# ==========================================================

SUBSAMPLE_WINDOWS = {
    "M6_pre": {
        "start": None,
        "end": "2019-12-31"
    },
    "M6_covid": {
        "start": "2020-01-01",
        "end": "2021-12-31"
    },
    "M6_post": {
        "start": "2022-01-01",
        "end": None
    }
}

print("\nSubmuestras definidas:")
for k, v in SUBSAMPLE_WINDOWS.items():
    print(f"{k}: start={v['start']} | end={v['end']}")


# ==========================================================
# 10.3 HELPERS DE FILTRADO TEMPORAL
# ==========================================================

def filter_panel_by_date(df, start=None, end=None, date_level=1):
    """
    Filtra un panel con MultiIndex [entity, date] por rango temporal.
    """
    out = df.copy()

    dates = pd.to_datetime(out.index.get_level_values(date_level))

    mask = pd.Series(True, index=out.index)

    if start is not None:
        start = pd.Timestamp(start)
        mask &= (dates >= start)

    if end is not None:
        end = pd.Timestamp(end)
        mask &= (dates <= end)

    out = out[mask.values].copy()
    return out


def print_subsample_basic_info(df_sub, label):
    """
    Resumen simple previo a dropna por modelo.
    """
    n_obs_raw = len(df_sub)
    n_firms_raw = df_sub.index.get_level_values(0).nunique()
    n_time_raw = df_sub.index.get_level_values(1).nunique()

    print(f"\n[{label}] Panel bruto")
    print(f"Observaciones brutas: {n_obs_raw}")
    print(f"Firmas brutas: {n_firms_raw}")
    print(f"Periodos brutos: {n_time_raw}")


# ==========================================================
# 10.4 FUNCIÓN DE ESTIMACIÓN DE M6 POR SUBMUESTRA
# ==========================================================

def run_final_model_subsamples(
    df,
    dep_var,
    model_name,
    exog_vars,
    subsample_windows,
    add_time_fe=False,
    battery_name="subsamples_m6"
):
    """
    Corre el mismo modelo final en distintas ventanas temporales.
    Recalcula la muestra efectiva dentro de cada submuestra.
    """

    results_dict = {}
    sample_dict = {}
    spec_dict = {}

    subsample_summary_dir = MODEL_SUMMARY_DIR / battery_name
    subsample_sample_dir = MODEL_SAMPLE_DIR / battery_name

    subsample_summary_dir.mkdir(parents=True, exist_ok=True)
    subsample_sample_dir.mkdir(parents=True, exist_ok=True)

    for subsample_name, window in subsample_windows.items():

        print("\n" + "="*70)
        print(f"{battery_name.upper()} | Estimando {subsample_name}")
        print("="*70)

        # 1) filtrar período
        df_sub = filter_panel_by_date(
            df,
            start=window["start"],
            end=window["end"]
        )

        print_subsample_basic_info(df_sub, subsample_name)

        # 2) estimar con muestra efectiva propia
        res, df_model = estimate_model(
            df_sub,
            dep_var,
            exog_vars,
            add_time_fe=add_time_fe
        )

        # 3) guardar resultados
        results_dict[subsample_name] = res
        sample_dict[subsample_name] = sample_info(df_model)
        spec_dict[subsample_name] = {
            "model_base": model_name,
            "dep_var": dep_var,
            "rhs_vars": " | ".join(exog_vars),
            "time_fe": add_time_fe,
            "firm_fe": True,
            "battery": battery_name,
            "subsample_start": window["start"],
            "subsample_end": window["end"],
        }

        print(f"\n[{subsample_name}] Muestra efectiva")
        print(f"Observaciones: {sample_dict[subsample_name]['n_obs']}")
        print(f"Firmas: {sample_dict[subsample_name]['n_firms']}")
        print(f"Periodos: {sample_dict[subsample_name]['n_time']}")

        # 4) guardar summary txt
        with open(
            subsample_summary_dir / f"{subsample_name}_{dep_var}_summary.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(str(res.summary))

        # 5) guardar muestra efectiva
        df_model.reset_index().to_parquet(
            subsample_sample_dir / f"{subsample_name}_{dep_var}_sample.parquet",
            index=False
        )

    return results_dict, sample_dict, spec_dict


# ==========================================================
# 10.5 CORRIDA DE M6 EN SUBMUESTRAS
# ==========================================================

results_m6_sub, sample_m6_sub, spec_m6_sub = run_final_model_subsamples(
    df=df,
    dep_var=FINAL_DEP_VAR,
    model_name=FINAL_MODEL_NAME,
    exog_vars=FINAL_VARS,
    subsample_windows=SUBSAMPLE_WINDOWS,
    add_time_fe=FINAL_TIME_FE,
    battery_name="subsamples_m6"
)


# ==========================================================
# 10.6 PAPER TABLE CONSOLIDADA
# ==========================================================

paper_m6_sub = build_paper_table(
    results_dict=results_m6_sub,
    sample_dict=sample_m6_sub,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_subsamples"
)

print("\nVista previa paper table:")
display(paper_m6_sub.head(30))


# ==========================================================
# 10.7 SAMPLE SUMMARY
# ==========================================================

sample_summary_m6_sub = build_sample_summary(
    sample_dict=sample_m6_sub,
    spec_dict=spec_m6_sub,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_subsamples"
)

print("\nVista previa sample summary:")
display(sample_summary_m6_sub)


# ==========================================================
# 10.8 RUN METADATA
# ==========================================================

run_meta_m6_sub = build_run_metadata(
    sample_dict=sample_m6_sub,
    spec_dict=spec_m6_sub,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_subsamples"
)

print("\nVista previa run metadata:")
display(run_meta_m6_sub)


# ==========================================================
# 10.9 TABLA RESUMEN DE COEFICIENTES CLAVE
# ==========================================================

def build_key_coef_comparison(results_dict, model_order=None):
    """
    Tabla compacta para comparar coeficientes, SE y p-values
    de todas las variables estimadas en las submuestras.
    """
    if model_order is None:
        model_order = list(results_dict.keys())

    all_vars = []
    for m in model_order:
        for v in results_dict[m].params.index.tolist():
            if v not in all_vars:
                all_vars.append(v)

    rows = []
    for var in all_vars:
        row = {"variable": var}
        for m in model_order:
            res = results_dict[m]
            if var in res.params.index:
                row[f"{m}_coef"] = res.params[var]
                row[f"{m}_se"] = res.std_errors[var]
                row[f"{m}_pval"] = res.pvalues[var]
            else:
                row[f"{m}_coef"] = np.nan
                row[f"{m}_se"] = np.nan
                row[f"{m}_pval"] = np.nan
        rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(TABLES_DIR / f"key_coef_compare_{FINAL_DEP_VAR}_m6_subsamples.csv", index=False)
    out.to_excel(TABLES_DIR / f"key_coef_compare_{FINAL_DEP_VAR}_m6_subsamples.xlsx", index=False)

    print("✅ Tabla compacta de coeficientes clave guardada")
    return out


key_coef_m6_sub = build_key_coef_comparison(
    results_dict=results_m6_sub,
    model_order=["M6_pre", "M6_covid", "M6_post"]
)

print("\nVista previa comparación de coeficientes:")
display(key_coef_m6_sub)


# ==========================================================
# 10.10 LOG FINAL
# ==========================================================

print("\n" + "="*70)
print("ARCHIVOS EXPORTADOS — M6 SUBMUESTRAS")
print("="*70)

print("\nPaper table:")
print(TABLES_DIR / f"paper_table_{FINAL_DEP_VAR}_m6_subsamples.csv")
print(TABLES_DIR / f"paper_table_{FINAL_DEP_VAR}_m6_subsamples.xlsx")

print("\nSample summary:")
print(TABLES_DIR / f"sample_summary_{FINAL_DEP_VAR}_m6_subsamples.csv")
print(TABLES_DIR / f"sample_summary_{FINAL_DEP_VAR}_m6_subsamples.xlsx")

print("\nRun metadata:")
print(RESULTS_DIR / f"run_metadata_{FINAL_DEP_VAR}_m6_subsamples.csv")
print(RESULTS_DIR / f"run_metadata_{FINAL_DEP_VAR}_m6_subsamples.xlsx")

print("\nCoeficientes clave:")
print(TABLES_DIR / f"key_coef_compare_{FINAL_DEP_VAR}_m6_subsamples.csv")
print(TABLES_DIR / f"key_coef_compare_{FINAL_DEP_VAR}_m6_subsamples.xlsx")

print("\nModel summaries:")
print(MODEL_SUMMARY_DIR / "subsamples_m6")

print("\nModel samples:")
print(MODEL_SAMPLE_DIR / "subsamples_m6")

ROBUSTEZ TEMPORAL — MODELO FINAL
Modelo: M6
Variable dependiente: oas_mean
Time FE: False
N regressors: 16
Regresores:
 - leverage
 - log_assets
 - cash_to_assets
 - current_ratio
 - interest_coverage
 - residual_maturity_mean
 - rollover_12m
 - ivol_252
 - policy_rate
 - term_spread_10y_2y
 - move_eom
 - mkt_ret
 - credit_level
 - credit_slope
 - mkt_ret_x_leverage
 - credit_level_x_rollover

Submuestras definidas:
M6_pre: start=None | end=2019-12-31
M6_covid: start=2020-01-01 | end=2021-12-31
M6_post: start=2022-01-01 | end=None

SUBSAMPLES_M6 | Estimando M6_pre

[M6_pre] Panel bruto
Observaciones brutas: 10341
Firmas brutas: 221
Periodos brutos: 60

[M6_pre] Muestra efectiva
Observaciones: 4111
Firmas: 141
Periodos: 37

SUBSAMPLES_M6 | Estimando M6_covid

[M6_covid] Panel bruto
Observaciones brutas: 5762
Firmas brutas: 253
Periodos brutos: 24

[M6_covid] Muestra efectiva
Observaciones: 3690
Firmas: 162
Periodos: 24

SUBSAMPLES_M6 | Estimando M6_post

[M6_post] Panel bruto
Observacio

,variable,M6_pre,M6_covid,M6_post
0,cash_to_assets,40.8337,-187.2128***,-44.6314
1,,(37.0286),(65.8279),(53.1359)
2,credit_level,-369.2743***,202.8937***,-9.5853***
3,,(66.9603),(47.0596),(1.8678)
4,credit_level_x_rollover,143.6322,-386.5860*,9.0467
5,,(298.4347),(215.7605),(14.6673)
6,credit_slope,-103.5877***,-212.1117***,-1.1288***
7,,(36.4402),(34.4099),(0.0787)
8,current_ratio,-3.5907,15.1917,4.8469
9,,(4.1873),(9.2881),(4.9751)


✅ Sample summary (m6_subsamples) guardado

Vista previa sample summary:


,n_obs,n_firms,n_time,model_base,dep_var,rhs_vars,time_fe,firm_fe,battery,subsample_start,subsample_end
M6_pre,4111,141,37,M6,oas_mean,leverage | log_assets | cash_to_assets | curre...,False,True,subsamples_m6,None,2019-12-31
M6_covid,3690,162,24,M6,oas_mean,leverage | log_assets | cash_to_assets | curre...,False,True,subsamples_m6,2020-01-01,2021-12-31
M6_post,7905,170,48,M6,oas_mean,leverage | log_assets | cash_to_assets | curre...,False,True,subsamples_m6,2022-01-01,None


✅ Metadata (m6_subsamples) guardada

Vista previa run metadata:


,model,dep_var,time_fe,rhs_vars,battery,n_obs,n_firms,n_time
0,M6_pre,oas_mean,False,leverage | log_assets | cash_to_assets | curre...,subsamples_m6,4111,141,37
1,M6_covid,oas_mean,False,leverage | log_assets | cash_to_assets | curre...,subsamples_m6,3690,162,24
2,M6_post,oas_mean,False,leverage | log_assets | cash_to_assets | curre...,subsamples_m6,7905,170,48


✅ Tabla compacta de coeficientes clave guardada

Vista previa comparación de coeficientes:


,variable,M6_pre_coef,M6_pre_se,M6_pre_pval,M6_covid_coef,M6_covid_se,M6_covid_pval,M6_post_coef,M6_post_se,M6_post_pval
0,leverage,-23.793492,31.214115,4.459461e-01,-71.906970,34.384950,3.657879e-02,24.997819,28.345071,3.778526e-01
1,log_assets,5.416152,7.736245,4.839055e-01,6.832690,15.028440,6.493887e-01,5.021909,7.804338,5.199340e-01
2,cash_to_assets,40.833655,37.028570,2.701982e-01,-187.212776,65.827924,4.481260e-03,-44.631399,53.135920,4.009637e-01
3,current_ratio,-3.590738,4.187345,3.912087e-01,15.191674,9.288129,1.020127e-01,4.846925,4.975081,3.299678e-01
4,interest_coverage,-0.049272,0.019541,1.172531e-02,0.140732,0.040751,5.600011e-04,-0.014880,0.022176,5.022321e-01
5,residual_maturity_mean,1.846826,0.877210,3.532470e-02,15.144965,2.474422,1.034826e-09,2.263078,0.981547,2.115805e-02
6,rollover_12m,-15.937970,22.067884,4.701991e-01,44.749833,33.487520,1.815333e-01,-11.522503,14.399637,4.236230e-01
7,ivol_252,321.466418,46.521112,5.620393e-12,207.305319,34.779079,2.760913e-09,212.846041,40.351564,1.365161e-07
8,policy_rate,10.166942,1.970628,2.601222e-07,-55.236658,3.910089,0.000000e+00,-7.776349,0.914926,0.000000e+00
9,term_spread_10y_2y,33.602442,3.462271,0.000000e+00,-88.547337,4.237665,0.000000e+00,-14.169356,2.964282,1.785112e-06



ARCHIVOS EXPORTADOS — M6 SUBMUESTRAS

Paper table:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\paper_table_oas_mean_m6_subsamples.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\paper_table_oas_mean_m6_subsamples.xlsx

Sample summary:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\sample_summary_oas_mean_m6_subsamples.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\sample_summary_oas_mean_m6_subsamples.xlsx

Run metadata:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\results\run_metadata_oas_mean_m6_subsamples.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\results\run_metadata_oas_mean_m6_subsamples.xlsx

Coeficientes clave:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\key_coef_compare_oas_mean_m6_subsamples.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\key_coef_compare_oas_mean_m6_subsamples.xlsx

11. Robustez de identificación — M6 baseline vs M6 con efectos fijos de tiempo

En esta sección se compara el modelo final M6 en dos versiones:

una especificación baseline con efectos fijos por firma y sin efectos fijos de tiempo;
una especificación alternativa con efectos fijos por firma y efectos fijos de tiempo.

El objetivo es evaluar hasta qué punto los resultados de M6 dependen de variación agregada común a todas las firmas en cada período, o si reflejan heterogeneidad firm-level más robusta.

La lógica econométrica del ejercicio es la siguiente. En la especificación sin efectos fijos de tiempo, los coeficientes se identifican a partir de la combinación de variación temporal agregada y variación transversal entre firmas. En cambio, al incorporar efectos fijos de tiempo, toda perturbación común a las firmas en un mismo período queda absorbida por las dummies temporales. De este modo, la identificación pasa a descansar en variación within adicional, especialmente en la heterogeneidad de exposición de las firmas frente a shocks agregados.

Este contraste es particularmente relevante en esta tesis porque la nueva estrategia de identificación busca distinguir explícitamente entre:

factores agregados de mercado y de crédito;
exposición diferencial de las firmas a dichos shocks;
características estructurales firm-level.

Por lo tanto, este test permite responder una pregunta central: si los coeficientes de M6 cambian sustancialmente al introducir efectos fijos de tiempo, ello sugiere que parte de su poder explicativo en la versión baseline provenía de shocks agregados comunes. En cambio, si los coeficientes asociados a interacciones o variables firm-level se mantienen relativamente estables, ello constituye evidencia más sólida de heterogeneidad estructural entre firmas.

En términos prácticos, esta sección:

estima M6 baseline y M6 con time fixed effects;
guarda los summaries de ambas regresiones;
exporta las muestras efectivas utilizadas;
construye una tabla comparativa tipo paper;
arma una tabla compacta de coeficientes, errores estándar y p-values;
deja trazabilidad para interpretar cambios en magnitud, signo y significancia.

La lectura de resultados debe enfocarse en cuatro dimensiones:

estabilidad de signos;
cambios en magnitud;
pérdida o persistencia de significancia;
diferencias en tamaño muestral o estructura de la muestra efectiva.

Si los factores agregados en nivel pierden significancia al incluir time FE, ese resultado no debe interpretarse automáticamente como una falla del modelo, sino como evidencia de que dichos factores capturan principalmente variación temporal común. En cambio, si sobreviven las interacciones entre shocks agregados y características firm-level, ello refuerza la interpretación estructural buscada en la tesis: no solo importan los shocks agregados, sino también la exposición diferencial de las firmas a esos shocks.

In [26]:
# ==========================================================
# 11. ROBUSTEZ — M6 BASELINE VS M6 + TIME FE
# ==========================================================

# ==========================================================
# 11.1 CONFIGURACIÓN DEL MODELO FINAL
# ==========================================================

FINAL_MODEL_NAME = "M6"
FINAL_MODEL_CONFIG = M6_CONFIG.copy()

FINAL_DEP_VAR = DEP_VAR_MAIN
FINAL_VARS = FINAL_MODEL_CONFIG["vars"]

print("=" * 70)
print("ROBUSTEZ DE IDENTIFICACIÓN — M6 BASELINE VS M6 + TIME FE")
print("=" * 70)
print("Modelo base:", FINAL_MODEL_NAME)
print("Variable dependiente:", FINAL_DEP_VAR)
print("Número de regresores:", len(FINAL_VARS))
print("Regresores:")
for v in FINAL_VARS:
    print(" -", v)


# ==========================================================
# 11.2 FUNCIÓN AUXILIAR — ESTIMACIÓN FLEXIBLE CON TIME FE
# ==========================================================

from linearmodels.panel import PanelOLS

def estimate_model_flexible(df, dep_var, exog_vars, add_time_fe=False, drop_absorbed=False):
    """
    Estima un modelo PanelOLS permitiendo opcionalmente dropear
    variables absorbidas por los efectos fijos.
    """
    df_model = prepare_model_data(df, dep_var, exog_vars)

    y = df_model[dep_var]
    X = df_model[exog_vars]

    model = PanelOLS(
        y,
        X,
        entity_effects=True,
        time_effects=add_time_fe,
        drop_absorbed=drop_absorbed
    )

    results = model.fit(
        cov_type="clustered",
        cluster_entity=True
    )

    # Variables efectivamente estimadas
    estimated_vars = list(results.params.index)

    # Variables absorbidas o excluidas
    absorbed_vars = [v for v in exog_vars if v not in estimated_vars]

    return results, df_model, estimated_vars, absorbed_vars


# ==========================================================
# 11.3 FUNCIÓN DE COMPARACIÓN BASELINE VS TIME FE
# ==========================================================

def run_time_fe_comparison(df, dep_var, model_name, exog_vars, battery_name="m6_time_fe_compare"):
    """
    Corre dos versiones del mismo modelo:
    - baseline: firm FE, sin time FE
    - time_fe: firm FE, con time FE y drop automático de absorbidas
    """

    results_dict = {}
    sample_dict = {}
    spec_dict = {}
    absorbed_info = []

    comparison_summary_dir = MODEL_SUMMARY_DIR / battery_name
    comparison_sample_dir = MODEL_SAMPLE_DIR / battery_name

    comparison_summary_dir.mkdir(parents=True, exist_ok=True)
    comparison_sample_dir.mkdir(parents=True, exist_ok=True)

    comparison_specs = {
        f"{model_name}_baseline": {
            "add_time_fe": False,
            "drop_absorbed": False
        },
        f"{model_name}_time_fe": {
            "add_time_fe": True,
            "drop_absorbed": True
        },
    }

    for model_label, cfg in comparison_specs.items():

        print("\n" + "=" * 70)
        print(f"{battery_name.upper()} | Estimando {model_label}")
        print("=" * 70)

        res, df_model, estimated_vars, absorbed_vars = estimate_model_flexible(
            df=df,
            dep_var=dep_var,
            exog_vars=exog_vars,
            add_time_fe=cfg["add_time_fe"],
            drop_absorbed=cfg["drop_absorbed"]
        )

        results_dict[model_label] = res
        sample_dict[model_label] = sample_info(df_model)
        spec_dict[model_label] = {
        "model_base": model_name,
        "dep_var": dep_var,
        "rhs_vars": " | ".join(estimated_vars),
        "rhs_vars_requested": " | ".join(exog_vars),
        "rhs_vars_estimated": " | ".join(estimated_vars),
        "time_fe": cfg["add_time_fe"],
        "firm_fe": True,
        "battery": battery_name,
        }           

        for v in exog_vars:
            absorbed_info.append({
                "model": model_label,
                "variable": v,
                "estimated": v in estimated_vars,
                "absorbed_or_dropped": v in absorbed_vars,
                "time_fe": cfg["add_time_fe"]
            })

        print(f"Observaciones: {sample_dict[model_label]['n_obs']}")
        print(f"Firmas: {sample_dict[model_label]['n_firms']}")
        print(f"Periodos: {sample_dict[model_label]['n_time']}")

        print("Variables estimadas:")
        for v in estimated_vars:
            print(" -", v)

        if absorbed_vars:
            print("Variables absorbidas/dropeadas:")
            for v in absorbed_vars:
                print(" -", v)

        # guardar summary txt
        with open(
            comparison_summary_dir / f"{model_label}_{dep_var}_summary.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(str(res.summary))

        # guardar muestra efectiva
        df_model.reset_index().to_parquet(
            comparison_sample_dir / f"{model_label}_{dep_var}_sample.parquet",
            index=False
        )

    absorbed_df = pd.DataFrame(absorbed_info)
    absorbed_df.to_csv(TABLES_DIR / f"absorbed_vars_{dep_var}_{battery_name}.csv", index=False)
    absorbed_df.to_excel(TABLES_DIR / f"absorbed_vars_{dep_var}_{battery_name}.xlsx", index=False)

    return results_dict, sample_dict, spec_dict, absorbed_df


# ==========================================================
# 11.4 CORRIDA DE M6 BASELINE Y M6 + TIME FE
# ==========================================================

results_m6_tfe, sample_m6_tfe, spec_m6_tfe, absorbed_m6_tfe = run_time_fe_comparison(
    df=df,
    dep_var=FINAL_DEP_VAR,
    model_name=FINAL_MODEL_NAME,
    exog_vars=FINAL_VARS,
    battery_name="m6_time_fe_compare"
)


# ==========================================================
# 11.5 PAPER TABLE COMPARATIVA
# ==========================================================

paper_m6_tfe = build_paper_table(
    results_dict=results_m6_tfe,
    sample_dict=sample_m6_tfe,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_baseline_vs_timefe"
)

print("\nVista previa paper table:")
display(paper_m6_tfe.head(40))


# ==========================================================
# 11.6 SAMPLE SUMMARY
# ==========================================================

sample_summary_m6_tfe = build_sample_summary(
    sample_dict=sample_m6_tfe,
    spec_dict=spec_m6_tfe,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_baseline_vs_timefe"
)

print("\nVista previa sample summary:")
display(sample_summary_m6_tfe)


# ==========================================================
# 11.7 RUN METADATA
# ==========================================================

run_meta_m6_tfe = build_run_metadata(
    sample_dict=sample_m6_tfe,
    spec_dict=spec_m6_tfe,
    dep_var=FINAL_DEP_VAR,
    suffix="m6_baseline_vs_timefe"
)

print("\nVista previa run metadata:")
display(run_meta_m6_tfe)


# ==========================================================
# 11.8 TABLA COMPACTA DE COEFICIENTES
# ==========================================================

def build_key_coef_comparison(results_dict, model_order=None, outname="key_coef_compare_m6_baseline_vs_timefe"):
    if model_order is None:
        model_order = list(results_dict.keys())

    all_vars = []
    for m in model_order:
        for v in results_dict[m].params.index.tolist():
            if v not in all_vars:
                all_vars.append(v)

    rows = []
    for var in all_vars:
        row = {"variable": var}
        for m in model_order:
            res = results_dict[m]
            if var in res.params.index:
                row[f"{m}_coef"] = res.params[var]
                row[f"{m}_se"] = res.std_errors[var]
                row[f"{m}_pval"] = res.pvalues[var]
            else:
                row[f"{m}_coef"] = np.nan
                row[f"{m}_se"] = np.nan
                row[f"{m}_pval"] = np.nan
        rows.append(row)

    out = pd.DataFrame(rows)

    out.to_csv(TABLES_DIR / f"{outname}_{FINAL_DEP_VAR}.csv", index=False)
    out.to_excel(TABLES_DIR / f"{outname}_{FINAL_DEP_VAR}.xlsx", index=False)

    print("Tabla compacta de coeficientes guardada")
    return out


key_coef_m6_tfe = build_key_coef_comparison(
    results_dict=results_m6_tfe,
    model_order=[f"{FINAL_MODEL_NAME}_baseline", f"{FINAL_MODEL_NAME}_time_fe"],
    outname="key_coef_compare_m6_baseline_vs_timefe"
)

print("\nVista previa comparación de coeficientes:")
display(key_coef_m6_tfe)


# ==========================================================
# 11.9 TABLA DE CAMBIOS RELATIVOS
# ==========================================================

def build_coef_change_table(results_dict, baseline_model, compare_model, outname="coef_change_m6_baseline_vs_timefe"):
    res_base = results_dict[baseline_model]
    res_comp = results_dict[compare_model]

    all_vars = sorted(set(res_base.params.index).union(set(res_comp.params.index)))

    rows = []
    for var in all_vars:
        base_coef = res_base.params[var] if var in res_base.params.index else np.nan
        comp_coef = res_comp.params[var] if var in res_comp.params.index else np.nan

        base_p = res_base.pvalues[var] if var in res_base.pvalues.index else np.nan
        comp_p = res_comp.pvalues[var] if var in res_comp.pvalues.index else np.nan

        delta_abs = comp_coef - base_coef if pd.notna(base_coef) and pd.notna(comp_coef) else np.nan

        if pd.notna(base_coef) and base_coef != 0 and pd.notna(comp_coef):
            delta_pct = (comp_coef - base_coef) / abs(base_coef)
        else:
            delta_pct = np.nan

        rows.append({
            "variable": var,
            f"{baseline_model}_coef": base_coef,
            f"{baseline_model}_pval": base_p,
            f"{compare_model}_coef": comp_coef,
            f"{compare_model}_pval": comp_p,
            "delta_abs": delta_abs,
            "delta_pct": delta_pct
        })

    out = pd.DataFrame(rows)

    out.to_csv(TABLES_DIR / f"{outname}_{FINAL_DEP_VAR}.csv", index=False)
    out.to_excel(TABLES_DIR / f"{outname}_{FINAL_DEP_VAR}.xlsx", index=False)

    print("Tabla de cambios relativos guardada")
    return out


coef_change_m6_tfe = build_coef_change_table(
    results_dict=results_m6_tfe,
    baseline_model=f"{FINAL_MODEL_NAME}_baseline",
    compare_model=f"{FINAL_MODEL_NAME}_time_fe",
    outname="coef_change_m6_baseline_vs_timefe"
)

print("\nVista previa cambios relativos:")
display(coef_change_m6_tfe)


# ==========================================================
# 11.10 VARIABLES ABSORBIDAS
# ==========================================================

print("\nVista previa variables absorbidas:")
display(absorbed_m6_tfe)


# ==========================================================
# 11.11 LOG FINAL
# ==========================================================

print("\n" + "=" * 70)
print("ARCHIVOS EXPORTADOS — M6 BASELINE VS M6 + TIME FE")
print("=" * 70)

print("\nPaper table:")
print(TABLES_DIR / f"paper_table_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.csv")
print(TABLES_DIR / f"paper_table_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.xlsx")

print("\nSample summary:")
print(TABLES_DIR / f"sample_summary_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.csv")
print(TABLES_DIR / f"sample_summary_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.xlsx")

print("\nRun metadata:")
print(RESULTS_DIR / f"run_metadata_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.csv")
print(RESULTS_DIR / f"run_metadata_{FINAL_DEP_VAR}_m6_baseline_vs_timefe.xlsx")

print("\nCoeficientes clave:")
print(TABLES_DIR / f"key_coef_compare_m6_baseline_vs_timefe_{FINAL_DEP_VAR}.csv")
print(TABLES_DIR / f"key_coef_compare_m6_baseline_vs_timefe_{FINAL_DEP_VAR}.xlsx")

print("\nCambios relativos:")
print(TABLES_DIR / f"coef_change_m6_baseline_vs_timefe_{FINAL_DEP_VAR}.csv")
print(TABLES_DIR / f"coef_change_m6_baseline_vs_timefe_{FINAL_DEP_VAR}.xlsx")

print("\nVariables absorbidas:")
print(TABLES_DIR / f"absorbed_vars_{FINAL_DEP_VAR}_m6_time_fe_compare.csv")
print(TABLES_DIR / f"absorbed_vars_{FINAL_DEP_VAR}_m6_time_fe_compare.xlsx")

print("\nModel summaries:")
print(MODEL_SUMMARY_DIR / "m6_time_fe_compare")

print("\nModel samples:")
print(MODEL_SAMPLE_DIR / "m6_time_fe_compare")

ROBUSTEZ DE IDENTIFICACIÓN — M6 BASELINE VS M6 + TIME FE
Modelo base: M6
Variable dependiente: oas_mean
Número de regresores: 16
Regresores:
 - leverage
 - log_assets
 - cash_to_assets
 - current_ratio
 - interest_coverage
 - residual_maturity_mean
 - rollover_12m
 - ivol_252
 - policy_rate
 - term_spread_10y_2y
 - move_eom
 - mkt_ret
 - credit_level
 - credit_slope
 - mkt_ret_x_leverage
 - credit_level_x_rollover

M6_TIME_FE_COMPARE | Estimando M6_baseline
Observaciones: 15706
Firmas: 170
Periodos: 109
Variables estimadas:
 - leverage
 - log_assets
 - cash_to_assets
 - current_ratio
 - interest_coverage
 - residual_maturity_mean
 - rollover_12m
 - ivol_252
 - policy_rate
 - term_spread_10y_2y
 - move_eom
 - mkt_ret
 - credit_level
 - credit_slope
 - mkt_ret_x_leverage
 - credit_level_x_rollover

M6_TIME_FE_COMPARE | Estimando M6_time_fe
Observaciones: 15706
Firmas: 170
Periodos: 109
Variables estimadas:
 - leverage
 - log_assets
 - cash_to_assets
 - current_ratio
 - interest_coverage


,variable,M6_baseline,M6_time_fe
0,cash_to_assets,-91.9819***,-45.4020
1,,(33.9717),(30.3467)
2,credit_level,0.3653,
3,,(2.2381),
4,credit_level_x_rollover,10.2145,8.5654
5,,(15.8878),(14.8558)
6,credit_slope,-0.9646***,
7,,(0.0793),
8,current_ratio,7.0605*,5.5043
9,,(4.0704),(3.6631)


✅ Sample summary (m6_baseline_vs_timefe) guardado

Vista previa sample summary:


,n_obs,n_firms,n_time,model_base,dep_var,rhs_vars,rhs_vars_requested,rhs_vars_estimated,time_fe,firm_fe,battery
M6_baseline,15706,170,109,M6,oas_mean,leverage | log_assets | cash_to_assets | curre...,leverage | log_assets | cash_to_assets | curre...,leverage | log_assets | cash_to_assets | curre...,False,True,m6_time_fe_compare
M6_time_fe,15706,170,109,M6,oas_mean,leverage | log_assets | cash_to_assets | curre...,leverage | log_assets | cash_to_assets | curre...,leverage | log_assets | cash_to_assets | curre...,True,True,m6_time_fe_compare


✅ Metadata (m6_baseline_vs_timefe) guardada

Vista previa run metadata:


...

Tabla de cambios relativos guardada

Vista previa cambios relativos:


,variable,M6_baseline_coef,M6_baseline_pval,M6_time_fe_coef,M6_time_fe_pval,delta_abs,delta_pct
0,cash_to_assets,-91.981932,6.784421e-03,-45.402023,1.346458e-01,46.579909,0.506403
1,credit_level,0.365261,8.703631e-01,NaN,NaN,NaN,NaN
2,credit_level_x_rollover,10.214542,5.202861e-01,8.565425,5.642380e-01,-1.649118,-0.161448
3,credit_slope,-0.964594,0.000000e+00,NaN,NaN,NaN,NaN
4,current_ratio,7.060510,8.283342e-02,5.504345,1.329506e-01,-1.556166,-0.220404
5,interest_coverage,0.025379,3.684400e-01,0.009442,6.784460e-01,-0.015938,-0.627975
6,ivol_252,257.116800,0.000000e+00,233.357400,7.044809e-11,-23.759400,-0.092407
7,leverage,26.715745,1.515436e-01,30.469752,1.144047e-01,3.754007,0.140517
8,log_assets,0.579135,9.270716e-01,12.797814,3.588389e-02,12.218679,21.098149
9,mkt_ret,-65.258301,1.524311e-05,NaN,NaN,NaN,NaN



Vista previa variables absorbidas:


,model,variable,estimated,absorbed_or_dropped,time_fe
0,M6_baseline,leverage,True,False,False
1,M6_baseline,log_assets,True,False,False
2,M6_baseline,cash_to_assets,True,False,False
3,M6_baseline,current_ratio,True,False,False
4,M6_baseline,interest_coverage,True,False,False
5,M6_baseline,residual_maturity_mean,True,False,False
6,M6_baseline,rollover_12m,True,False,False
7,M6_baseline,ivol_252,True,False,False
8,M6_baseline,policy_rate,True,False,False
9,M6_baseline,term_spread_10y_2y,True,False,False



ARCHIVOS EXPORTADOS — M6 BASELINE VS M6 + TIME FE

Paper table:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\paper_table_oas_mean_m6_baseline_vs_timefe.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\paper_table_oas_mean_m6_baseline_vs_timefe.xlsx

Sample summary:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\sample_summary_oas_mean_m6_baseline_vs_timefe.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\sample_summary_oas_mean_m6_baseline_vs_timefe.xlsx

Run metadata:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\results\run_metadata_oas_mean_m6_baseline_vs_timefe.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\results\run_metadata_oas_mean_m6_baseline_vs_timefe.xlsx

Coeficientes clave:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\key_coef_compare_m6_baseline_vs_timefe_oas_mean.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\T

# 12. Diagnóstico de multicolinealidad — VIF

En esta sección se evalúa la presencia de multicolinealidad entre los regresores del modelo final y, en particular, entre los bloques de variables agregadas de mercado, crédito y controles macroeconómicos.

El objetivo no es eliminar mecánicamente variables por un umbral arbitrario, sino identificar si existen redundancias conceptuales o problemas de colinealidad que dificulten la interpretación de los coeficientes.

La lógica de este ejercicio es especialmente importante en esta tesis porque la estrategia empírica distingue entre:

shocks agregados de mercado;
shocks y niveles agregados de crédito;
dinámica temporal de dichos factores;
heterogeneidad firm-level en la exposición a esos shocks.

Dado que varias de estas variables están relacionadas entre sí por construcción económica, es esperable observar cierto grado de correlación. Sin embargo, cuando esa relación se vuelve excesiva, pueden aparecer problemas de identificación empírica:

errores estándar inflados;
pérdida de significancia individual;
inestabilidad de coeficientes;
sensibilidad excesiva a pequeños cambios de especificación.

Para evaluar esto se calcula el Variance Inflation Factor (VIF) sobre el conjunto de regresores del modelo. El VIF mide cuánto aumenta la varianza estimada de un coeficiente debido a su colinealidad con el resto de las variables explicativas.

La interpretación debe hacerse con cautela:

VIF bajos indican ausencia de problemas relevantes;
VIF moderados pueden ser consistentes con relaciones económicas razonables;
VIF altos sugieren posible redundancia estadística o conceptual.

En esta tesis, el foco principal del diagnóstico está en:

variables macroeconómicas;
factores agregados de crédito;
shock agregado de mercado;
interacciones relevantes del modelo final.

Además del VIF, esta sección exporta una tabla con el diagnóstico para facilitar su revisión, documentación y eventual uso en los apéndices de la tesis.

In [27]:
# ==========================================================
# 12. DIAGNÓSTICO DE MULTICOLINEALIDAD — VIF
# ==========================================================

import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ==========================================================
# 12.1 CONFIGURACIÓN
# ==========================================================

# Modelo final a diagnosticar
FINAL_MODEL_NAME = "M6"
FINAL_MODEL_CONFIG = M6_CONFIG.copy()

FINAL_DEP_VAR = DEP_VAR_MAIN
FINAL_VARS = FINAL_MODEL_CONFIG["vars"]

print("=" * 70)
print("DIAGNÓSTICO DE MULTICOLINEALIDAD — VIF")
print("=" * 70)
print("Modelo:", FINAL_MODEL_NAME)
print("Variable dependiente:", FINAL_DEP_VAR)
print("Número de regresores solicitados:", len(FINAL_VARS))
print("Regresores:")
for v in FINAL_VARS:
    print(" -", v)


# ==========================================================
# 12.2 PREPARAR MUESTRA DEL MODELO FINAL
# ==========================================================

df_vif = prepare_model_data(df, FINAL_DEP_VAR, FINAL_VARS).copy()

print("\nMuestra preparada para VIF")
print("Observaciones:", len(df_vif))
print("Firmas:", df_vif.index.get_level_values(0).nunique())
print("Periodos:", df_vif.index.get_level_values(1).nunique())


# ==========================================================
# 12.3 DEFINIR BLOQUES DE VARIABLES
# ==========================================================

macro_candidates = [v for v in MACRO_VARS if v in FINAL_VARS]
market_candidates = [v for v in [MARKET_SHOCK] if v in FINAL_VARS]

credit_candidates = [
    v for v in [
        "credit_level",
        "credit_slope",
        "credit_tightening_shock",
        "credit_tightening_shock_l1",
        "credit_market_level_log",
        "credit_market_change"
    ]
    if v in FINAL_VARS
]

interaction_candidates = [
    v for v in [
        "mkt_ret_x_leverage",
        "mkt_ret_x_market_power",
        "credit_level_x_rollover",
        "credit_level_x_market_power",
        "credit_tightening_shock_x_rollover",
        "credit_tightening_shock_x_market_power",
        "credit_tightening_shock_l1_x_rollover",
        "credit_tightening_shock_l1_x_market_power",
        "credit_market_level_log_x_rollover",
        "credit_market_level_log_x_market_power"
    ]
    if v in FINAL_VARS
]

firm_level_candidates = [
    v for v in FINAL_VARS
    if v not in set(macro_candidates + market_candidates + credit_candidates + interaction_candidates)
]

print("\nBloques detectados:")
print("Macro:", macro_candidates)
print("Mercado:", market_candidates)
print("Crédito:", credit_candidates)
print("Interacciones:", interaction_candidates)
print("Firm-level / otros:", firm_level_candidates)


# ==========================================================
# 12.4 FUNCIÓN PARA CALCULAR VIF
# ==========================================================

def compute_vif_table(df_input, vars_list, add_constant=True):
    """
    Calcula tabla de VIF para una lista de variables.
    Limpia tipos de datos, infs y columnas sin variación.
    """
    vars_present = [v for v in vars_list if v in df_input.columns]

    if len(vars_present) == 0:
        return pd.DataFrame(columns=["variable", "vif"])

    X = df_input[vars_present].copy()

    # Forzar numérico columna por columna
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    # Reemplazar infinitos
    X = X.replace([np.inf, -np.inf], np.nan)

    # Eliminar filas con NA luego de coerción
    n_before = len(X)
    X = X.dropna()
    n_after = len(X)

    if n_after < n_before:
        print(f"\nFilas eliminadas por NA/inf en VIF: {n_before - n_after}")

    if len(X) == 0:
        print("\nNo quedaron observaciones válidas para calcular VIF")
        return pd.DataFrame(columns=["variable", "vif"])

    # Eliminar columnas constantes o sin variación
    nunique = X.nunique(dropna=False)
    constant_like = nunique[nunique <= 1].index.tolist()
    if constant_like:
        print("\nVariables eliminadas por falta de variación:", constant_like)
        X = X.drop(columns=constant_like)

    # Revalidar columnas
    vars_present = X.columns.tolist()

    if len(vars_present) == 0:
        print("\nNo quedaron variables con variación para calcular VIF")
        return pd.DataFrame(columns=["variable", "vif"])

    # Chequeo final de dtypes
    bad_dtypes = X.dtypes[X.dtypes == "object"]
    if len(bad_dtypes) > 0:
        print("\nColumnas aún no numéricas:")
        print(bad_dtypes)
        raise TypeError("Persisten columnas no numéricas en la matriz para VIF")

    # Agregar constante
    if add_constant:
        X = X.copy()
        X["const"] = 1.0

    vif_rows = []
    for i, col in enumerate(X.columns):
        if col == "const":
            continue
        vif_value = variance_inflation_factor(X.values.astype(float), i)
        vif_rows.append({
            "variable": col,
            "vif": vif_value
        })

    out = pd.DataFrame(vif_rows).sort_values("vif", ascending=False).reset_index(drop=True)
    return out


# ==========================================================
# 12.4.B DIAGNÓSTICO DE TIPOS
# ==========================================================

print("\nDtypes de variables del modelo final:")
dtype_check = df_vif[FINAL_VARS].dtypes.sort_index()
print(dtype_check)

non_numeric_cols = [
    c for c in FINAL_VARS
    if c in df_vif.columns and not pd.api.types.is_numeric_dtype(df_vif[c])
]

print("\nColumnas no numéricas detectadas:")
print(non_numeric_cols if non_numeric_cols else "Ninguna")

# ==========================================================
# 12.5 TABLA VIF — MODELO COMPLETO
# ==========================================================

vif_full = compute_vif_table(df_vif, FINAL_VARS)

print("\nVIF — modelo completo")
display(vif_full)


# ==========================================================
# 12.6 TABLAS VIF POR BLOQUE
# ==========================================================

vif_macro_market_credit = compute_vif_table(
    df_vif,
    macro_candidates + market_candidates + credit_candidates
)

vif_interactions = compute_vif_table(
    df_vif,
    interaction_candidates
)

vif_firm_level = compute_vif_table(
    df_vif,
    firm_level_candidates
)

print("\nVIF — macro + mercado + crédito")
display(vif_macro_market_credit)

print("\nVIF — interacciones")
display(vif_interactions)

print("\nVIF — firm-level / otros")
display(vif_firm_level)


# ==========================================================
# 12.7 CLASIFICACIÓN DE SEVERIDAD
# ==========================================================

def classify_vif(v):
    if pd.isna(v):
        return "NA"
    elif v < 5:
        return "Bajo"
    elif v < 10:
        return "Moderado"
    else:
        return "Alto"

if len(vif_full) > 0:
    vif_full["severity"] = vif_full["vif"].apply(classify_vif)

if len(vif_macro_market_credit) > 0:
    vif_macro_market_credit["severity"] = vif_macro_market_credit["vif"].apply(classify_vif)

if len(vif_interactions) > 0:
    vif_interactions["severity"] = vif_interactions["vif"].apply(classify_vif)

if len(vif_firm_level) > 0:
    vif_firm_level["severity"] = vif_firm_level["vif"].apply(classify_vif)


# ==========================================================
# 12.8 TABLA RESUMEN PARA DECISIÓN
# ==========================================================

vif_decision = vif_full.copy()
if len(vif_decision) > 0:
    vif_decision["recommendation"] = np.select(
        [
            vif_decision["vif"] < 5,
            (vif_decision["vif"] >= 5) & (vif_decision["vif"] < 10),
            vif_decision["vif"] >= 10
        ],
        [
            "Mantener",
            "Revisar interpretación conjunta",
            "Revisar posible redundancia"
        ],
        default="Revisar"
    )

print("\nTabla resumen para decisión")
display(vif_decision)


# ==========================================================
# 12.9 EXPORTS
# ==========================================================

vif_full.to_csv(TABLES_DIR / f"vif_full_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv", index=False)
vif_full.to_excel(TABLES_DIR / f"vif_full_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx", index=False)

vif_macro_market_credit.to_csv(
    TABLES_DIR / f"vif_macro_market_credit_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv",
    index=False
)
vif_macro_market_credit.to_excel(
    TABLES_DIR / f"vif_macro_market_credit_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx",
    index=False
)

vif_interactions.to_csv(
    TABLES_DIR / f"vif_interactions_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv",
    index=False
)
vif_interactions.to_excel(
    TABLES_DIR / f"vif_interactions_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx",
    index=False
)

vif_firm_level.to_csv(
    TABLES_DIR / f"vif_firm_level_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv",
    index=False
)
vif_firm_level.to_excel(
    TABLES_DIR / f"vif_firm_level_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx",
    index=False
)

vif_decision.to_csv(
    TABLES_DIR / f"vif_decision_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv",
    index=False
)
vif_decision.to_excel(
    TABLES_DIR / f"vif_decision_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx",
    index=False
)

print("\nArchivos exportados:")
print(TABLES_DIR / f"vif_full_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv")
print(TABLES_DIR / f"vif_full_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx")
print(TABLES_DIR / f"vif_macro_market_credit_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv")
print(TABLES_DIR / f"vif_macro_market_credit_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx")
print(TABLES_DIR / f"vif_interactions_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv")
print(TABLES_DIR / f"vif_interactions_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx")
print(TABLES_DIR / f"vif_firm_level_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv")
print(TABLES_DIR / f"vif_firm_level_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx")
print(TABLES_DIR / f"vif_decision_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.csv")
print(TABLES_DIR / f"vif_decision_{FINAL_DEP_VAR}_{FINAL_MODEL_NAME}.xlsx")

DIAGNÓSTICO DE MULTICOLINEALIDAD — VIF
Modelo: M6
Variable dependiente: oas_mean
Número de regresores solicitados: 16
Regresores:
 - leverage
 - log_assets
 - cash_to_assets
 - current_ratio
 - interest_coverage
 - residual_maturity_mean
 - rollover_12m
 - ivol_252
 - policy_rate
 - term_spread_10y_2y
 - move_eom
 - mkt_ret
 - credit_level
 - credit_slope
 - mkt_ret_x_leverage
 - credit_level_x_rollover

Muestra preparada para VIF
Observaciones: 15706
Firmas: 170
Periodos: 109

Bloques detectados:
Macro: ['policy_rate', 'term_spread_10y_2y', 'move_eom']
Mercado: ['mkt_ret']
Crédito: ['credit_level', 'credit_slope']
Interacciones: ['mkt_ret_x_leverage', 'credit_level_x_rollover']
Firm-level / otros: ['leverage', 'log_assets', 'cash_to_assets', 'current_ratio', 'interest_coverage', 'residual_maturity_mean', 'rollover_12m', 'ivol_252']

Dtypes de variables del modelo final:
cash_to_assets             Float64
credit_level               float64
credit_level_x_rollover    Float64
credit_slop

,variable,vif
0,mkt_ret,7.383940
1,mkt_ret_x_leverage,7.278429
2,term_spread_10y_2y,2.753922
3,policy_rate,2.555049
4,move_eom,2.326435
5,credit_level,1.900458
6,credit_level_x_rollover,1.865494
7,current_ratio,1.572448
8,cash_to_assets,1.454753
9,leverage,1.200343



VIF — macro + mercado + crédito


,variable,vif
0,term_spread_10y_2y,2.651813
1,policy_rate,2.370740
2,move_eom,2.191356
3,mkt_ret,1.137051
4,credit_level,1.035357
5,credit_slope,1.003068



VIF — interacciones


...


VIF — firm-level / otros


,variable,vif
0,current_ratio,1.572091
1,cash_to_assets,1.434684
2,interest_coverage,1.166622
3,leverage,1.146201
4,log_assets,1.096977
5,rollover_12m,1.056780
6,residual_maturity_mean,1.030202
7,ivol_252,1.029364



Tabla resumen para decisión


,variable,vif,severity,recommendation
0,mkt_ret,7.383940,Moderado,Revisar interpretación conjunta
1,mkt_ret_x_leverage,7.278429,Moderado,Revisar interpretación conjunta
2,term_spread_10y_2y,2.753922,Bajo,Mantener
3,policy_rate,2.555049,Bajo,Mantener
4,move_eom,2.326435,Bajo,Mantener
5,credit_level,1.900458,Bajo,Mantener
6,credit_level_x_rollover,1.865494,Bajo,Mantener
7,current_ratio,1.572448,Bajo,Mantener
8,cash_to_assets,1.454753,Bajo,Mantener
9,leverage,1.200343,Bajo,Mantener



Archivos exportados:
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_full_oas_mean_M6.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_full_oas_mean_M6.xlsx
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_macro_market_credit_oas_mean_M6.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_macro_market_credit_oas_mean_M6.xlsx
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_interactions_oas_mean_M6.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_interactions_oas_mean_M6.xlsx
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_firm_level_oas_mean_M6.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_firm_level_oas_mean_M6.xlsx
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\outputs\tables\vif_decision_oas_mean_M6.csv
C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\ou

# 9.X Robustez — Driscoll–Kraay

Como ejercicio de robustez adicional, se reestiman los modelos de la batería principal
(M0–M6) utilizando errores estándar **Driscoll–Kraay**.

Esta corrección es apropiada en paneles con dimensión temporal relativamente grande,
ya que permite realizar inferencia robusta frente a:

- heterocedasticidad,
- autocorrelación temporal,
- dependencia transversal entre firmas.

La especificación econométrica de cada modelo se mantiene idéntica a la utilizada en la
estimación principal. En consecuencia, cualquier cambio en la significancia estadística
de los coeficientes responde exclusivamente a la modificación de la matriz de
varianza-covarianza y no a cambios en la muestra o en el conjunto de regresores.

Los resultados se exportan en subcarpetas separadas, replicando la lógica de guardado del
runner principal, para facilitar la comparación con la batería base estimada con errores
estándar agrupados por firma.

In [29]:
# ==========================================================
# 9.X ROBUSTEZ — DRISCOLL-KRAAY (BATERÍA MAIN)
# ==========================================================

from linearmodels.panel import PanelOLS
import pandas as pd
import numpy as np

# ----------------------------------------------------------
# Parámetros DK
# ----------------------------------------------------------
DK_KERNEL = "bartlett"
DK_BANDWIDTH = 4   # mensual; podés probar después con 6

# ----------------------------------------------------------
# Carpetas específicas DK
# ----------------------------------------------------------
DK_RESULTS_DIR = RESULTS_DIR / "driscoll_kraay"
DK_TABLES_DIR = TABLES_DIR / "driscoll_kraay"

DK_MODEL_SUMMARY_DIR = DK_RESULTS_DIR / "model_summaries"
DK_MODEL_SAMPLE_DIR = DK_RESULTS_DIR / "model_samples"

DK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
DK_MODEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
DK_MODEL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 9.X.1 FUNCIONES DK
# ==========================================================

def run_panel_model_dk(df, dep_var, exog_vars, add_time_fe=False, kernel="bartlett", bandwidth=4):
    """
    Estima PanelOLS con:
    - FE firma siempre
    - FE tiempo opcional
    - errores estándar Driscoll-Kraay
    """
    y = df[dep_var]
    X = df[exog_vars]

    model = PanelOLS(
        y,
        X,
        entity_effects=True,
        time_effects=add_time_fe
    )

    res = model.fit(
        cov_type="kernel",
        kernel=kernel,
        bandwidth=bandwidth
    )

    return res


def estimate_model_dk(df, dep_var, exog_vars, add_time_fe=False, kernel="bartlett", bandwidth=4):
    """
    Replica la lógica base:
    - prepara muestra efectiva
    - estima con DK
    """
    df_model = prepare_model_data(df, dep_var, exog_vars)

    res = run_panel_model_dk(
        df_model,
        dep_var,
        exog_vars,
        add_time_fe=add_time_fe,
        kernel=kernel,
        bandwidth=bandwidth
    )

    return res, df_model


def run_model_battery_dk(df, dep_var, model_specs, battery_name, kernel="bartlett", bandwidth=4):
    """
    Replica run_model_battery() pero con Driscoll-Kraay.
    """
    results_dict = {}
    sample_dict = {}
    spec_dict = {}

    battery_summary_dir = DK_MODEL_SUMMARY_DIR / battery_name
    battery_sample_dir = DK_MODEL_SAMPLE_DIR / battery_name

    battery_summary_dir.mkdir(parents=True, exist_ok=True)
    battery_sample_dir.mkdir(parents=True, exist_ok=True)

    for model_name, config in model_specs.items():

        print(f"\n==============================")
        print(f"DK | {battery_name.upper()} | Estimando {model_name}")
        print(f"==============================")

        vars_model = config["vars"]
        time_fe = config["time_fe"]

        res, df_model = estimate_model_dk(
            df=df,
            dep_var=dep_var,
            exog_vars=vars_model,
            add_time_fe=time_fe,
            kernel=kernel,
            bandwidth=bandwidth
        )

        results_dict[model_name] = res
        sample_dict[model_name] = sample_info(df_model)
        spec_dict[model_name] = {
            "dep_var": dep_var,
            "rhs_vars": " | ".join(vars_model),
            "time_fe": time_fe,
            "firm_fe": True,
            "battery": battery_name,
            "cov_type": "Driscoll-Kraay",
            "kernel": kernel,
            "bandwidth": bandwidth,
        }

        print(f"Observaciones: {sample_dict[model_name]['n_obs']}")
        print(f"Firmas: {sample_dict[model_name]['n_firms']}")
        print(f"Periodos: {sample_dict[model_name]['n_time']}")

        # Guardar summary
        with open(battery_summary_dir / f"{model_name}_{dep_var}_summary_dk.txt", "w", encoding="utf-8") as f:
            f.write(str(res.summary))

        # Guardar muestra efectiva
        df_model.to_parquet(
            battery_sample_dir / f"{model_name}_{dep_var}_sample_dk.parquet",
            index=False
        )

    return results_dict, sample_dict, spec_dict


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

# ==========================================================
# 9.X.2 ESTIMACIÓN DK — BATERÍA PRINCIPAL
# ==========================================================

results_main_dk, sample_main_dk, spec_main_dk = run_model_battery_dk(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_MAIN,
    battery_name="main",
    kernel=DK_KERNEL,
    bandwidth=DK_BANDWIDTH
)

# ==========================================================
# 9.X.3 EXPORT TABLAS DE MUESTRA
# ==========================================================

sample_main_dk_df = pd.DataFrame(sample_main_dk).T
spec_main_dk_df = pd.DataFrame(spec_main_dk).T
sample_summary_main_dk = sample_main_dk_df.join(spec_main_dk_df)

sample_summary_main_dk.to_csv(
    DK_TABLES_DIR / f"sample_summary_main_{DEP_VAR_MAIN}_dk.csv",
    index=True
)
sample_summary_main_dk.to_excel(
    DK_TABLES_DIR / f"sample_summary_main_{DEP_VAR_MAIN}_dk.xlsx",
    index=True
)

print("\n✅ sample_summary_main_dk guardado")

# ==========================================================
# 9.X.4 EXPORT TABLA CONSOLIDADA DE COEFICIENTES
# ==========================================================

coef_tables_main_dk = []

for model_name, res in results_main_dk.items():
    tmp = pd.DataFrame({
        "variable": res.params.index,
        f"{model_name}_coef": res.params.values,
        f"{model_name}_se_dk": res.std_errors.values,
        f"{model_name}_t_dk": res.tstats.values,
        f"{model_name}_p_dk": res.pvalues.values,
    })
    coef_tables_main_dk.append(tmp)

coef_main_dk_df = coef_tables_main_dk[0].copy()
for t in coef_tables_main_dk[1:]:
    coef_main_dk_df = coef_main_dk_df.merge(t, on="variable", how="outer")

coef_main_dk_df.to_csv(
    DK_TABLES_DIR / f"coefficients_main_{DEP_VAR_MAIN}_dk.csv",
    index=False
)
coef_main_dk_df.to_excel(
    DK_TABLES_DIR / f"coefficients_main_{DEP_VAR_MAIN}_dk.xlsx",
    index=False
)

print("✅ coefficients_main_dk guardado")

# ==========================================================
# 9.X.5 TABLA PAPER-STYLE MAIN DK
# ==========================================================

paper_rows_main_dk = []

all_vars_main_dk = []
for model_name, res in results_main_dk.items():
    for v in res.params.index:
        if v not in all_vars_main_dk:
            all_vars_main_dk.append(v)

for var in all_vars_main_dk:
    row_coef = {"variable": var}
    row_se = {"variable": ""}

    for model_name, res in results_main_dk.items():
        if var in res.params.index:
            coef = res.params[var]
            se = res.std_errors[var]
            p = res.pvalues[var]

            row_coef[model_name] = f"{coef:.4f}{stars(p)}"
            row_se[model_name] = f"({se:.4f})"
        else:
            row_coef[model_name] = ""
            row_se[model_name] = ""

    paper_rows_main_dk.append(row_coef)
    paper_rows_main_dk.append(row_se)

stats_labels = {
    "n_obs": "Observaciones",
    "n_firms": "Firmas",
    "n_time": "Periodos",
}

for key, label in stats_labels.items():
    row = {"variable": label}
    for model_name in results_main_dk.keys():
        row[model_name] = sample_main_dk[model_name][key]
    paper_rows_main_dk.append(row)

paper_table_main_dk = pd.DataFrame(paper_rows_main_dk)

paper_table_main_dk.to_csv(
    DK_TABLES_DIR / f"paper_table_main_{DEP_VAR_MAIN}_dk.csv",
    index=False
)
paper_table_main_dk.to_excel(
    DK_TABLES_DIR / f"paper_table_main_{DEP_VAR_MAIN}_dk.xlsx",
    index=False
)

print("✅ paper_table_main_dk guardado")
print("✅ Robustez Driscoll-Kraay (MAIN) finalizada")


DK | MAIN | Estimando M0
Observaciones: 15706
Firmas: 170
Periodos: 109

DK | MAIN | Estimando M1
Observaciones: 15706
Firmas: 170
Periodos: 109

DK | MAIN | Estimando M2
Observaciones: 15706
Firmas: 170
Periodos: 109

DK | MAIN | Estimando M3
Observaciones: 13969
Firmas: 168
Periodos: 108

DK | MAIN | Estimando M4
Observaciones: 15706
Firmas: 170
Periodos: 109

DK | MAIN | Estimando M5
Observaciones: 13969
Firmas: 168
Periodos: 108

DK | MAIN | Estimando M6
Observaciones: 15706
Firmas: 170
Periodos: 109

✅ sample_summary_main_dk guardado
✅ coefficients_main_dk guardado
✅ paper_table_main_dk guardado
✅ Robustez Driscoll-Kraay (MAIN) finalizada


# 9.X Robustez — Efectos fijos Sector × Tiempo

Como ejercicio de robustez adicional, se reestiman los modelos de la batería principal
(M0–M6) incorporando efectos fijos de **sector × tiempo**.

Esta especificación permite absorber shocks sectoriales mensuales no observados que afectan
simultáneamente a todas las firmas dentro de un mismo sector, tales como:

- cambios regulatorios específicos,
- shocks de demanda o de costos sectoriales,
- episodios de estrés financiero concentrados en determinadas industrias,
- variaciones en condiciones competitivas o tecnológicas comunes al sector.

La motivación es reforzar la identificación de los efectos de las variables firm-level y de los
shocks agregados, evitando atribuirles variación que en realidad responde a dinámicas
sectoriales contemporáneas.

La especificación mantiene efectos fijos por firma y agrega una dimensión adicional de control
mediante efectos fijos de sector × tiempo. Los errores estándar se mantienen agrupados por
firma para conservar comparabilidad con la estimación principal.

Los resultados se exportan en subcarpetas separadas para facilitar su comparación con la
batería base.

In [31]:
# ==========================================================
# 9.X ROBUSTEZ — SECTOR × TIME FE (BATERÍA MAIN)
# ==========================================================

from linearmodels.panel import PanelOLS
import pandas as pd
import numpy as np

# ----------------------------------------------------------
# Detectar variable sectorial
# ----------------------------------------------------------
SECTOR_CANDIDATES = [
    "gics_sector",
    "sector",
    "gics_sector_name",
    "gics_sector_group",
    "sector_name"
]

SECTOR_VAR = None
for c in SECTOR_CANDIDATES:
    if c in df.columns:
        SECTOR_VAR = c
        break

if SECTOR_VAR is None:
    raise ValueError(
        f"No encontré una variable sectorial en el panel. Revisá estas opciones: {SECTOR_CANDIDATES}"
    )

print(f"✅ Variable sectorial detectada: {SECTOR_VAR}")

# ----------------------------------------------------------
# Carpetas específicas Sector×Time FE
# ----------------------------------------------------------
SXT_RESULTS_DIR = RESULTS_DIR / "sector_time_fe"
SXT_TABLES_DIR = TABLES_DIR / "sector_time_fe"

SXT_MODEL_SUMMARY_DIR = SXT_RESULTS_DIR / "model_summaries"
SXT_MODEL_SAMPLE_DIR = SXT_RESULTS_DIR / "model_samples"

SXT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SXT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
SXT_MODEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
SXT_MODEL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 9.X.1 FUNCIONES
# ==========================================================

def prepare_model_data_sector_time(df, dep_var, exog_vars, sector_var):
    """
    Prepara muestra efectiva para modelos con FE firma + sector×tiempo.
    """
    needed_cols = [dep_var] + exog_vars + [sector_var]
    tmp = df[needed_cols].copy().dropna()

    if not isinstance(tmp.index, pd.MultiIndex):
        raise ValueError("El DataFrame debe tener MultiIndex (firma, fecha).")

    if tmp.index.nlevels != 2:
        raise ValueError("El MultiIndex debe tener exactamente 2 niveles: entidad y tiempo.")

    # importante: ordenar índice para evitar problemas de alineación
    tmp = tmp.sort_index()

    return tmp


def build_sector_time_effect(df_model, sector_var):
    """
    Construye FE sector×tiempo con el mismo MultiIndex que df_model.
    Devuelve una DataFrame numérica de una columna.
    """
    # mismo índice panel
    idx = df_model.index

    # segundo nivel del índice = tiempo
    time_vals = pd.Index(idx.get_level_values(1))
    time_str = pd.to_datetime(time_vals).to_period("M").astype(str)

    # sector
    sector_str = df_model[sector_var].astype(str)

    # etiqueta sector×mes
    sector_time_label = sector_str.values + "__" + np.asarray(time_str)

    # convertir a códigos numéricos manteniendo exactamente el mismo índice
    sector_time_codes = pd.Categorical(sector_time_label).codes

    other_fx = pd.DataFrame(
        {"sector_time_fe": sector_time_codes},
        index=idx
    )

    return other_fx


def run_panel_model_sector_time(df_model, dep_var, exog_vars, sector_var):
    """
    Estima PanelOLS con:
    - FE firma
    - FE sector×tiempo vía other_effects
    - EE cluster por firma
    """
    y = df_model[dep_var]
    X = df_model[exog_vars]

    other_fx = build_sector_time_effect(df_model, sector_var)

    # chequeo defensivo
    if not other_fx.index.equals(y.index):
        raise ValueError("other_effects no quedó alineado con y.index")

    model = PanelOLS(
        y,
        X,
        entity_effects=True,
        other_effects=other_fx,
        drop_absorbed=True
    )

    res = model.fit(
        cov_type="clustered",
        cluster_entity=True
    )

    return res


def estimate_model_sector_time(df, dep_var, exog_vars, sector_var):
    """
    Prepara muestra efectiva y estima FE firma + sector×tiempo.
    """
    df_model = prepare_model_data_sector_time(df, dep_var, exog_vars, sector_var)

    res = run_panel_model_sector_time(
        df_model=df_model,
        dep_var=dep_var,
        exog_vars=exog_vars,
        sector_var=sector_var
    )

    return res, df_model


def run_model_battery_sector_time(df, dep_var, model_specs, battery_name, sector_var):
    """
    Replica la lógica de run_model_battery para FE firma + sector×tiempo.
    """
    results_dict = {}
    sample_dict = {}
    spec_dict = {}

    battery_summary_dir = SXT_MODEL_SUMMARY_DIR / battery_name
    battery_sample_dir = SXT_MODEL_SAMPLE_DIR / battery_name

    battery_summary_dir.mkdir(parents=True, exist_ok=True)
    battery_sample_dir.mkdir(parents=True, exist_ok=True)

    for model_name, config in model_specs.items():

        print("\n==============================")
        print(f"SECTOR×TIME FE | {battery_name.upper()} | Estimando {model_name}")
        print("==============================")

        vars_model = config["vars"]

        res, df_model = estimate_model_sector_time(
            df=df,
            dep_var=dep_var,
            exog_vars=vars_model,
            sector_var=sector_var
        )

        results_dict[model_name] = res
        sample_dict[model_name] = sample_info(df_model)
        spec_dict[model_name] = {
            "dep_var": dep_var,
            "rhs_vars": " | ".join(vars_model),
            "firm_fe": True,
            "sector_time_fe": True,
            "sector_var": sector_var,
            "battery": battery_name,
            "cov_type": "Clustered by firm",
        }

        print(f"Observaciones: {sample_dict[model_name]['n_obs']}")
        print(f"Firmas: {sample_dict[model_name]['n_firms']}")
        print(f"Periodos: {sample_dict[model_name]['n_time']}")

        # Guardar summary
        with open(battery_summary_dir / f"{model_name}_{dep_var}_summary_sector_time_fe.txt", "w", encoding="utf-8") as f:
            f.write(str(res.summary))

        # Guardar muestra efectiva
        df_model.to_parquet(
            battery_sample_dir / f"{model_name}_{dep_var}_sample_sector_time_fe.parquet",
            index=False
        )

    return results_dict, sample_dict, spec_dict


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

# ==========================================================
# 9.X.2 ESTIMACIÓN — BATERÍA PRINCIPAL
# ==========================================================

results_main_sxt, sample_main_sxt, spec_main_sxt = run_model_battery_sector_time(
    df=df,
    dep_var=DEP_VAR_MAIN,
    model_specs=MODEL_SPECS_MAIN,
    battery_name="main",
    sector_var=SECTOR_VAR
)

# ==========================================================
# 9.X.3 EXPORT TABLAS DE MUESTRA
# ==========================================================

sample_main_sxt_df = pd.DataFrame(sample_main_sxt).T
spec_main_sxt_df = pd.DataFrame(spec_main_sxt).T
sample_summary_main_sxt = sample_main_sxt_df.join(spec_main_sxt_df)

sample_summary_main_sxt.to_csv(
    SXT_TABLES_DIR / f"sample_summary_main_{DEP_VAR_MAIN}_sector_time_fe.csv",
    index=True
)
sample_summary_main_sxt.to_excel(
    SXT_TABLES_DIR / f"sample_summary_main_{DEP_VAR_MAIN}_sector_time_fe.xlsx",
    index=True
)

print("\n✅ sample_summary_main_sector_time_fe guardado")

# ==========================================================
# 9.X.4 EXPORT TABLA CONSOLIDADA DE COEFICIENTES
# ==========================================================

coef_tables_main_sxt = []

for model_name, res in results_main_sxt.items():
    tmp = pd.DataFrame({
        "variable": res.params.index,
        f"{model_name}_coef": res.params.values,
        f"{model_name}_se": res.std_errors.values,
        f"{model_name}_t": res.tstats.values,
        f"{model_name}_p": res.pvalues.values,
    })
    coef_tables_main_sxt.append(tmp)

coef_main_sxt_df = coef_tables_main_sxt[0].copy()
for t in coef_tables_main_sxt[1:]:
    coef_main_sxt_df = coef_main_sxt_df.merge(t, on="variable", how="outer")

coef_main_sxt_df.to_csv(
    SXT_TABLES_DIR / f"coefficients_main_{DEP_VAR_MAIN}_sector_time_fe.csv",
    index=False
)
coef_main_sxt_df.to_excel(
    SXT_TABLES_DIR / f"coefficients_main_{DEP_VAR_MAIN}_sector_time_fe.xlsx",
    index=False
)

print("✅ coefficients_main_sector_time_fe guardado")

# ==========================================================
# 9.X.5 TABLA PAPER-STYLE
# ==========================================================

paper_rows_main_sxt = []

all_vars_main_sxt = []
for model_name, res in results_main_sxt.items():
    for v in res.params.index:
        if v not in all_vars_main_sxt:
            all_vars_main_sxt.append(v)

for var in all_vars_main_sxt:
    row_coef = {"variable": var}
    row_se = {"variable": ""}

    for model_name, res in results_main_sxt.items():
        if var in res.params.index:
            coef = res.params[var]
            se = res.std_errors[var]
            p = res.pvalues[var]

            row_coef[model_name] = f"{coef:.4f}{stars(p)}"
            row_se[model_name] = f"({se:.4f})"
        else:
            row_coef[model_name] = ""
            row_se[model_name] = ""

    paper_rows_main_sxt.append(row_coef)
    paper_rows_main_sxt.append(row_se)

stats_labels = {
    "n_obs": "Observaciones",
    "n_firms": "Firmas",
    "n_time": "Periodos",
}

for key, label in stats_labels.items():
    row = {"variable": label}
    for model_name in results_main_sxt.keys():
        row[model_name] = sample_main_sxt[model_name][key]
    paper_rows_main_sxt.append(row)

paper_table_main_sxt = pd.DataFrame(paper_rows_main_sxt)

paper_table_main_sxt.to_csv(
    SXT_TABLES_DIR / f"paper_table_main_{DEP_VAR_MAIN}_sector_time_fe.csv",
    index=False
)
paper_table_main_sxt.to_excel(
    SXT_TABLES_DIR / f"paper_table_main_{DEP_VAR_MAIN}_sector_time_fe.xlsx",
    index=False
)

print("✅ paper_table_main_sector_time_fe guardado")
print("✅ Robustez Sector × Time FE (MAIN) finalizada")

✅ Variable sectorial detectada: sector

SECTOR×TIME FE | MAIN | Estimando M0
Observaciones: 15706
Firmas: 170
Periodos: 109

SECTOR×TIME FE | MAIN | Estimando M1
Observaciones: 15706
Firmas: 170
Periodos: 109

SECTOR×TIME FE | MAIN | Estimando M2
Observaciones: 15706
Firmas: 170
Periodos: 109

SECTOR×TIME FE | MAIN | Estimando M3
Observaciones: 13969
Firmas: 168
Periodos: 108

SECTOR×TIME FE | MAIN | Estimando M4
Observaciones: 15706
Firmas: 170
Periodos: 109

SECTOR×TIME FE | MAIN | Estimando M5
Observaciones: 13969
Firmas: 168
Periodos: 108

SECTOR×TIME FE | MAIN | Estimando M6
Observaciones: 15706
Firmas: 170
Periodos: 109

✅ sample_summary_main_sector_time_fe guardado
✅ coefficients_main_sector_time_fe guardado
✅ paper_table_main_sector_time_fe guardado
✅ Robustez Sector × Time FE (MAIN) finalizada
